# **Zusammenfassung der Arbeit auf dem Porto-Seguro-Datensatz**: EDA, Modell-Experimente, Hyperparameter-Tuning, Ensembles, und Modellvalidierung auf den Testdaten.

Dieses Notebook fasst die wichtigsten Erkenntnisse und Schritte der Analyse auf dem Porto-Seguro-Datensatz zusammen. Enthalten sind dabei Analysen aus der explorativen Datenanalyse, Modellvergleiche und -experimente, Evaluation von Feature-Repräsentationen sowie Fehlwert-Behandlungsmethoden sowie die spätere Untersuchung von HistGradientBoosting inklusive Hyperparameter-Suche und Ensemblemethoden. Abschließend erfolgt die Evaluation auf dem Testdatensatzsplit.

## Inhaltsverzeichnis

1. [Vorbereitung](#1-vorbereitung)
2. [EDA](#2-eda)
3. [Modell-Experimente und Vergleiche](#3-modellexperimente-und-vergleiche)
   - Linearität und Metriken
   - ps_calc_*-Features: Vergleichstest
   - Preprocessing
   - Isolation Forest
   - Manipulation des Klassenungleichgewichts
   - Isolation Score als Feature
   - Kategoriale Repräsentationen
   - Breiter Modellvergleich
   - Strategien zum Umgang mit Fehlwerten
4. [Zwischenfazit der EDA](#4-zwischenfazit-der-eda)
5. [Hyperparameteroptimierung (HPO)
](#5-hyperparameteroptimierung-(HPO)-und-Trainingsprozess
)
6. [Ensemblemethoden](#6-ensemblemethoden)
   - Durchschnitt, Rang-Durchschnitt und gewichteter Rang
   - Teilstichproben-Ensembles
   - Stacking mit logistischer Regression als Metalerner
   - Greedy Ensemble
7. [Finaler interner Test](#7-finaler-interner-test)
8. [Gesamtfazit](#8-gesamtfazit)

## 1. Vorbereitung

In [ ]:
# Seed für Reproduzierbarkeit und Target als Variable definieren
RANDOM_STATE = 42
TARGET = "target"

### 1.1 Importe

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import itertools
import math
import optuna
from scipy.stats import rankdata
from scipy.optimize import minimize
from catboost import CatBoostClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingClassifier, 
    IsolationForest,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss,
    RocCurveDisplay,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight


pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind")

### 1.2 Projektpfade und Daten laden

In [ ]:
# Projektstamm anhand der pyproject.toml suchen
ROOT = Path.cwd().resolve()

while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "openml" / "porto-seguro"
OUTPUT_DIR = ROOT / "results" / "eda_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR

In [ ]:
df = pd.read_parquet(DATA_DIR / "dataset_processed.parquet")

train_idx = np.load(DATA_DIR / "train_idx.npy")
val_idx = np.load(DATA_DIR / "val_idx.npy")
test_idx = np.load(DATA_DIR / "test_idx.npy")

train_df = df.iloc[train_idx].reset_index(drop=True)
validation_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("Gesamt:", df.shape, 
      "Train:", train_df.shape, 
      "Validation:", validation_df.shape,
      "Test:", test_df.shape
)

Erkenntnis: der vollständige Datensatz umfasst 595.212 Beobachtungen
und 58 Spalten. Die Aufteilung enthält 476.168 Trainings- sowie jeweils 59.522
Validierungs- und Testbeobachtungen.

## 2. EDA

### 2.1 Integritätsprüfung

In [ ]:
feature_cols = [col for col in df.columns if col != TARGET]

integrity_summary = pd.Series({
    "Beobachtungen": len(df),
    "Spalten": df.shape[1],
    "Features": len(feature_cols),
    "Doppelte Zeilen": df.duplicated().sum(),
    "Doppelte Feature-Zeilen": df[feature_cols].duplicated().sum(),
    "Fehlende Werte gesamt": df.isna().sum().sum(),
    "Features mit Missing Values": df[feature_cols].isna().any().sum(),
})

integrity_summary

In [ ]:
split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Beobachtungen": [len(train_df), len(validation_df), len(test_df)],
    "Positive Fälle": [
        train_df[TARGET].sum(),
        validation_df[TARGET].sum(),
        test_df[TARGET].sum(),
    ],
    "Positivrate (%)": [
        train_df[TARGET].mean() * 100,
        validation_df[TARGET].mean() * 100,
        test_df[TARGET].mean() * 100,
    ],
})

split_summary

Kurzfazit: in der ursprünglichen Prüfung wurden keine vollständigen Duplikate
gefunden. 13 Features enthalten fehlende Werte. Die Positivrate liegt in allen
drei Splits bei ungefähr 3,64 %, die Klassenverteilung wurde beim Split also erhalten.

### 2.2 Zielvariable

In [ ]:
target_distribution = (
    df[TARGET]
    .value_counts()
    .sort_index()
    .rename("Anzahl")
    .to_frame()
)

target_distribution["Anteil (%)"] = (
    target_distribution["Anzahl"] / len(df) * 100
)

target_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.barplot(
    x=target_distribution.index.astype(str),
    y=target_distribution["Anteil (%)"],
    ax=ax,
)

ax.set(
    title="Verteilung der Zielvariable",
    xlabel="Target",
    ylabel="Anteil (%)",
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3)

plt.tight_layout()
plt.show()

Erkenntnis: da nur 3,64 % der Beobachtungen zur positiven Klasse gehören, ist Akkuratheit allein später keine sinnvolle Hauptmetrik für die Optimierung.

### 2.3 Fehlende Werte

In [ ]:
missing_summary = (
    pd.DataFrame({
        "Anzahl fehlend": df[feature_cols].isna().sum(),
        "Fehlender Anteil": df[feature_cols].isna().mean(),
    })
    .query("`Anzahl fehlend` > 0")
    .sort_values("Fehlender Anteil", ascending=False)
)

missing_summary.head(10)

In [ ]:
plot_data = missing_summary.sort_values("Fehlender Anteil")

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    plot_data.index,
    plot_data["Fehlender Anteil"] * 100,
)

ax.set(
    title="Missing Values nach Feature",
    xlabel="Fehlender Anteil (%)",
    ylabel="Feature",
)

plt.tight_layout()
plt.show()

Die Missing Values konzentrieren sich auf wenige Features. Besonders auffällig
waren unter anderem ps_car_03_cat, ps_car_05_cat, ps_reg_03
und ps_car_14. Ein Löschen unvollständiger Zeilen mit df.dropna() wäre deshalb
ungeeignet.

#### Missingness und Target

In [ ]:
top_missing = (
    train_df[feature_cols]
    .isna()
    .mean()
    .nlargest(5)
    .index
)

missing_target_rows = []

for feature in top_missing:
    is_missing = train_df[feature].isna()

    missing_target_rows.append({
        "Feature": feature,
        "Missing-Anteil (%)": is_missing.mean() * 100,
        "Schadenquote fehlend (%)": train_df.loc[is_missing, TARGET].mean() * 100,
        "Schadenquote vorhanden (%)": train_df.loc[~is_missing, TARGET].mean() * 100,
    })

missing_target_table = pd.DataFrame(missing_target_rows)
missing_target_table

In [ ]:
missing_count = train_df[feature_cols].isna().sum(axis=1)

missing_count_table = (
    pd.DataFrame({
        "missing_count": missing_count,
        TARGET: train_df[TARGET],
    })
    .assign(
        missing_group=lambda x: x["missing_count"].where(
            x["missing_count"] <= 4, ">=5"
        )
    )
    .groupby("missing_group", observed=True)[TARGET]
    .agg(["count", "mean"])
)

missing_count_table["Schadenquote (%)"] = missing_count_table["mean"] * 100
missing_count_table

Kurzfazit: Missingness enthält selbst eine zielvariablen-Assoziation. Im ersten EDA-Teil zeigte sich z.B. bei ps_car_07_cat eine deutlich höhere Schadenquote bei einem fehlendem
Wert. Die Richtung ist aber je nach Feature unterschiedlich. Daher sollten Fehlwerte aber nicht sofort median-imputiert oder anders behandelt werden bevor nicht alle Optionen zur möglichen Integration der Fehlwertstruktur in die späteren Modele ausgetestet wurden.

### 2.4 Struktur ausgewählter Binärfeatures

In [ ]:
binary_group = [
    "ps_ind_06_bin",
    "ps_ind_07_bin",
    "ps_ind_08_bin",
    "ps_ind_09_bin",
]

train_df[binary_group].sum(axis=1).value_counts().sort_index()

Für diese vier Features gilt im Trainingsdatensatz vollständig: ps_ind_06_bin + ps_ind_07_bin + ps_ind_08_bin + ps_ind_09_bin = 1, d.h. pro Zeile ist jeweils ein Merkmal aktiv. Ein Feature kann daher aus allen anderen abgeleitet werden. Für lineare Modelle sollte deshalb
eine Referenzvariable weggelassen werden, um Multi-Kollinearität zu vermeiden.

In [ ]:
binary_features = [
    col for col in train_df.columns
    if col.endswith("_bin")
]

binary_target_rows = []

for feature in binary_features:
    rates = train_df.groupby(feature, observed=True)[TARGET].mean()

    if 0 in rates.index and 1 in rates.index:
        binary_target_rows.append({
            "Feature": feature,
            "Schadenquote 0 (%)": rates.loc[0] * 100,
            "Schadenquote 1 (%)": rates.loc[1] * 100,
            "Differenz (pp)": (rates.loc[1] - rates.loc[0]) * 100,
        })

binary_target = (
    pd.DataFrame(binary_target_rows)
    .assign(abs_diff=lambda x: x["Differenz (pp)"].abs())
    .sort_values("abs_diff", ascending=False)
    .drop(columns="abs_diff")
)

binary_target.head(10)

Die größten univariaten Unterschiede traten unter anderem bei ps_ind_17_bin, ps_ind_12_bin, ps_ind_07_bin und ps_ind_06_bin auf. Insgesamt bleibt die einzelne Feature-Target-Assoziation absolut gesehen aber eher schwach.

## 3. Modellexperimente und Vergleiche

In dieser Phase ging es um das Kennenlernen verschiedener Machine-Learning-Ansätze auf dem Porto-Seguro-Datensatz sowie nachfolgend um die Beantwortung von Fragen zur Datenrepräsentation und Modellwahl. Dieser Abschnitt fasst wesentliche Erkenntnisse daraus zusammen.

### 3.1 Preprocessing

In [ ]:
drop_cols = {"id", TARGET, "ps_ind_09_bin"}

model_features = [
    col for col in train_df.columns
    if col not in drop_cols
]

cat_features = [
    col for col in model_features
    if col.endswith("_cat")
]

num_features = [
    col for col in model_features
    if col not in cat_features
]

print("Features:", len(model_features))
print("Kategorial:", len(cat_features))
print("Sonstige:", len(num_features))

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features),
])

### 3.2 Linearität: Logistische Regression vs. Decision Tree

In [ ]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

In [ ]:
lr_model.fit(train_df[model_features], train_df[TARGET])
tree_model.fit(train_df[model_features], train_df[TARGET])

y_val = validation_df[TARGET]

lr_prob = lr_model.predict_proba(validation_df[model_features])[:, 1]
tree_prob = tree_model.predict_proba(validation_df[model_features])[:, 1]

linear_results = pd.DataFrame({
    "Modell": ["Logistische Regression", "Decision Tree"],
    "ROC-AUC": [
        roc_auc_score(y_val, lr_prob),
        roc_auc_score(y_val, tree_prob),
    ],
})

linear_results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_predictions(
    y_val, lr_prob, name="Logistische Regression", ax=ax
)

RocCurveDisplay.from_predictions(
    y_val, tree_prob, name="Decision Tree", ax=ax
)

ax.plot([0, 1], [0, 1], "--", linewidth=1)
ax.set_title("ROC-Kurven auf dem Validation-Split")

plt.tight_layout()
plt.show()

Bisheriges Ergebnis: die logistische Regression erreichte eine ROC-AUC von
0,6145, der begrenzte Decision Tree 0,5946. Nichtlinearität allein führt
also nicht automatisch zu einem besseren Modell. Dies motiviert später den Einsatz auch von Entscheidungsbaum-Ensembles, welche die größere Flexibilität mit mehr Robustheit verbinden.

### 3.3 Metrik-Falle bei unausgeglichenen Klassen

In [ ]:
perceptron = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Perceptron(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])

mlp = Pipeline([
    ("preprocessor", preprocessor),
    ("model", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        max_iter=20,
        early_stopping=True,
        random_state=RANDOM_STATE,
    )),
])

In [ ]:
perceptron.fit(train_df[model_features], train_df[TARGET])
mlp.fit(train_df[model_features], train_df[TARGET])

dummy_pred = np.zeros(len(validation_df), dtype=int)
perceptron_pred = perceptron.predict(validation_df[model_features])
mlp_pred = mlp.predict(validation_df[model_features])
mlp_prob = mlp.predict_proba(validation_df[model_features])[:, 1]

In [ ]:
metric_rows = []

for name, pred in [
    ("Dummy: immer 0", dummy_pred),
    ("Perzeptron", perceptron_pred),
    ("MLP", mlp_pred),
]:
    metric_rows.append({
        "Modell": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
    })

metric_table = pd.DataFrame(metric_rows)
metric_table

In [ ]:
print(
    "MLP ROC-AUC:",
    round(roc_auc_score(y_val, mlp_prob), 4)
)

Bisheriges Ergebnis: Selbst ein Dummy, der immer Klasse 0 vorhersagt,
erreicht etwa 96,36 % Accuracy. Das ungewichtete MLP kam ebenfalls auf
96,36 % Accuracy, hatte aber bei der Standardentscheidung praktisch keinen
Nutzen für die positive Klasse. Die ROC-AUC von rund 0,6085 zeigt dagegen,
dass trotzdem etwas Ranking-Signal vorhanden war.

Für die folgenden Experimente stehen deshalb ROC-AUC und ergänzend PR-AUC im
Vordergrund.

### 3.4 ps_calc_*-Features: Vergleichstest mit ihnen, nur mit ihnen und ohne sie

In [ ]:
calc_cols = [
    col for col in train_df.columns
    if col.startswith("ps_calc_")
]

non_calc_cols = [
    col for col in train_df.columns
    if col not in set(calc_cols) | {"id", TARGET, "ps_ind_09_bin"}
]

all_cols = non_calc_cols + calc_cols

print(
    "Alle:", len(all_cols),
    "| ps_calc:", len(calc_cols),
    "| ohne ps_calc:", len(non_calc_cols),
)

In [ ]:
def make_preprocessor(columns, scale=True, dense=False):
    cats = [col for col in columns if col.endswith("_cat")]
    nums = [col for col in columns if col not in cats]

    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer([
        ("num", Pipeline(num_steps), nums),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                drop="if_binary",
                sparse_output=not dense,
            )),
        ]), cats),
    ])

In [ ]:
feature_sets = {
    "NUR ps_calc-*": calc_cols,
    "OHNE ps_calc-*": non_calc_cols,
    "ALLE Features": all_cols,
}

ablation_rows = []

for set_name, columns in feature_sets.items():
    lr = Pipeline([
        ("prep", make_preprocessor(columns, scale=True)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])

    lr.fit(train_df[columns], train_df[TARGET])
    prob = lr.predict_proba(validation_df[columns])[:, 1]

    ablation_rows.append({
        "Feature-Set": set_name,
        "Modell": "Logistische Regression",
        "ROC-AUC": roc_auc_score(y_val, prob),
    })

In [ ]:
for set_name, columns in feature_sets.items():
    hgb = Pipeline([
        ("prep", make_preprocessor(columns, scale=False, dense=True)),
        ("model", HistGradientBoostingClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])

    hgb.fit(train_df[columns], train_df[TARGET])
    prob = hgb.predict_proba(validation_df[columns])[:, 1]

    ablation_rows.append({
        "Feature-Set": set_name,
        "Modell": "HistGradientBoosting",
        "ROC-AUC": roc_auc_score(y_val, prob),
    })

In [ ]:
ablation_results = pd.DataFrame(ablation_rows)

ablation_table = ablation_results.pivot(
    index="Feature-Set",
    columns="Modell",
    values="ROC-AUC",
)

ablation_table.round(4)

Bisheriges Ergebnis:

| Feature-Set | Logistische Regression | HistGradientBoosting |
|---|---:|---:|
| nur ps_calc_* | 0,4970 | 0,5049 |
| ohne ps_calc_* | 0,6148 | 0,6299 |
| alle Features | 0,6145 | 0,6294 |

Die ps_calc_\*-Features liefern isoliert praktisch kein Zielvariablensignal: die ROC-AUC lag fast auf dem Zufallsniveau. Das Hinzufügen zu den übrigen Features verbessert die beiden Baselines ebenfalls nicht, mit ihnen schnitten die verglichenen Modelle sogar leicht schlechter ab. Für spätere Modelle ist ein Feature-Set ohne ps_calc_* daher eine sinnvolle Vergleichsvariante.

### 3.5 Sensitivität gegenüber Preprocessing

In [ ]:
X_train_raw = train_df[non_calc_cols].copy()
X_val_raw = validation_df[non_calc_cols].copy()

# A: fehlende Werte als -1, sonst unverändert
X_train_a = X_train_raw.fillna(-1).to_numpy()
X_val_a = X_val_raw.fillna(-1).to_numpy()

# B: Median-Imputation
median_imputer = SimpleImputer(strategy="median")
X_train_b = median_imputer.fit_transform(X_train_raw)
X_val_b = median_imputer.transform(X_val_raw)

# C: Trennung von numerischen und kategorialen Features
clean_preprocessor = make_preprocessor(
    non_calc_cols,
    scale=True,
)

X_train_c = clean_preprocessor.fit_transform(X_train_raw)
X_val_c = clean_preprocessor.transform(X_val_raw)

In [ ]:
preprocessing_scenarios = {
    "A: Raw / -1": (X_train_a, X_val_a),
    "B: Median": (X_train_b, X_val_b),
    "C: OHE + Skalierung": (X_train_c, X_val_c),
}

neg_count, pos_count = np.bincount(train_df[TARGET].to_numpy())
scale_pos_weight = neg_count / pos_count

In [ ]:
prep_rows = []

for scenario, (X_tr, X_va) in preprocessing_scenarios.items():
    models = {
        "Logistische Regression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            max_iter=100,
            early_stopping=True,
            random_state=RANDOM_STATE,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=100,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbose=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100,
            learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            eval_metric="auc",
        ),
    }

    for model_name, model in models.items():
        model.fit(X_tr, train_df[TARGET])
        prob = model.predict_proba(X_va)[:, 1]

        prep_rows.append({
            "Szenario": scenario,
            "Modell": model_name,
            "ROC-AUC": roc_auc_score(y_val, prob),
        })

In [ ]:
preprocessing_results = (
    pd.DataFrame(prep_rows)
    .pivot(
        index="Modell",
        columns="Szenario",
        values="ROC-AUC",
    )
)

preprocessing_results.round(4)

Wie sich zeigte, war das half das vollständige Preprocessing v.a. den skalen- bzw. repräsentationssensitiven Modellen: der logistische Regression von 0,6100 auf 0,6148, dem MLP von 0,5936 auf 0,6203, während baumbasierte Ensembles wie LightGBM und XGBoost stabil bei ca. 0,63 verblieben. Das spricht dafür die Vorverarbeitung modell-abhängig zu testen.

### 3.6 Isolation Forest: Anomaliescore und Schadenquote

In [ ]:
iso_features = [
    col for col in train_df.columns
    if col not in {"id", TARGET}
]

iso_cat = [col for col in iso_features if col.endswith("_cat")]
iso_num = [col for col in iso_features if col not in iso_cat]

iso_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(
        strategy="median",
        add_indicator=True,
    ), iso_num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value=-1,
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            dtype=np.float32,
        )),
    ]), iso_cat),
])

In [ ]:
X_train_iso = iso_preprocessor.fit_transform(
    train_df[iso_features]
)

X_val_iso = iso_preprocessor.transform(
    validation_df[iso_features]
)

iso_forest = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

iso_forest.fit(X_train_iso)

In [ ]:
train_anomaly = -iso_forest.score_samples(X_train_iso)
val_anomaly = -iso_forest.score_samples(X_val_iso)

edges = np.quantile(
    train_anomaly,
    np.linspace(0, 1, 11),
)

edges[0] = -np.inf
edges[-1] = np.inf

In [ ]:
def anomaly_table(scores, target, split):
    temp = pd.DataFrame({
        "score": scores,
        TARGET: target.to_numpy(),
    })

    temp["Dezil"] = pd.cut(
        temp["score"],
        bins=edges,
        labels=False,
        include_lowest=True,
    ) + 1

    result = (
        temp.groupby("Dezil", observed=True)[TARGET]
        .agg(["count", "sum", "mean"])
        .reset_index()
    )

    result["Schadenquote (%)"] = result["mean"] * 100
    result["Split"] = split
    return result

In [ ]:
anomaly_results = pd.concat([
    anomaly_table(
        train_anomaly,
        train_df[TARGET],
        "Training",
    ),
    anomaly_table(
        val_anomaly,
        validation_df[TARGET],
        "Validation",
    ),
])

anomaly_results

Interpretation: im Validation-Split stieg die Schadenquote vom niedrigsten
Anomalie-Dezil mit rund 2,96 % auf rund 5,63 % im höchsten Dezil.
Der unüberwacht berechnete Anomaliescore scheint sich damit in dem gemeldeten Schadensrisiko widerzuspiegeln.

### 3.7 Manipulation des Klassenungleichgewichts

In [ ]:
X_train_res = train_df[non_calc_cols].copy()
X_val_res = validation_df[non_calc_cols].copy()

res_imputer = SimpleImputer(strategy="median")

X_train_res = res_imputer.fit_transform(X_train_res)
X_val_res = res_imputer.transform(X_val_res)

y_train = train_df[TARGET].to_numpy()
y_val_array = validation_df[TARGET].to_numpy()

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_smote, y_smote = smote.fit_resample(
    X_train_res,
    y_train,
)

undersampler = RandomUnderSampler(
    random_state=RANDOM_STATE
)
X_under, y_under = undersampler.fit_resample(
    X_train_res,
    y_train,
)

print("Original:", X_train_res.shape)
print("SMOTE:", X_smote.shape)
print("Undersampling:", X_under.shape)

In [ ]:
resampling_sets = {
    "Baseline": (X_train_res, y_train, {}),
    "Class Weights": (
        X_train_res,
        y_train,
        {"class_weight": "balanced"},
    ),
    "SMOTE": (X_smote, y_smote, {}),
    "Undersampling": (X_under, y_under, {}),
}

resampling_rows = []

In [ ]:
for strategy, (X_tr, y_tr, params) in resampling_sets.items():
    start = time.time()

    model = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
        verbose=-1,
        **params,
    )

    model.fit(X_tr, y_tr)
    prob = model.predict_proba(X_val_res)[:, 1]

    resampling_rows.append({
        "Strategie": strategy,
        "ROC-AUC": roc_auc_score(y_val_array, prob),
        "PR-AUC": average_precision_score(y_val_array, prob),
        "Trainingszeit (s)": time.time() - start,
    })

In [ ]:
resampling_results = (
    pd.DataFrame(resampling_rows)
    .sort_values("ROC-AUC", ascending=False)
)

resampling_results.round(4)

Interpretation: im ursprünglichen Vergleich brachte Synthetic Minority Oversampling Technique keinen Vorteil. Bei LightGBM fiel die ROC-AUC von etwa 0,6319 auf 0,6047 ab. Random
Undersampling (1:1) blieb erstaunlicherweise deutlich näher an der Baseline, während das Argument Class Weights die ROC-AUC ebenfalls nicht verbessern konnte. D.h. Resampling als solches nicht unbedingt die optimale Lösung für den Umgang mit dem Klassenungleichgewicht. 

### 3.8 Isolation Score als zusätzliches Feature

Der vorherige Isolation-Forest-Versuch zeigte, dass hohe Anomaliescores mit
einer höheren Schadenquote zusammenhängen. Als nächstes wurde geprüft, ob dieser
Score einem überwachten Modell tatsächlich zusätzliche Vorhersageinformation
liefert. Zu diesem Zwecke wurden die Baseline-Modelle einmal mit und einmal ohne den Isolationsscore trainiert.

In [ ]:
train_anomaly_score = train_anomaly
validation_anomaly_score = val_anomaly

In [ ]:
y_tr = train_df[TARGET].values.astype(np.int64)
y_va = validation_df[TARGET].values.astype(np.int64)

feature_cols = [
    c for c in train_df.columns
    if c not in {"id", TARGET, "ps_ind_09_bin"}
    and not c.startswith("ps_calc_")
]

In [ ]:
imp = SimpleImputer(strategy="median")

X_tr_base = imp.fit_transform(
    train_df[feature_cols]
)

X_va_base = imp.transform(
    validation_df[feature_cols]
)

In [ ]:
X_tr_aug = np.column_stack([
    X_tr_base,
    train_anomaly_score,
])

X_va_aug = np.column_stack([
    X_va_base,
    validation_anomaly_score,
])

In [ ]:
scale_pos = (
    len(y_tr) - np.sum(y_tr)
) / np.sum(y_tr)

scale_pos

In [ ]:
models = {
    "Logistische Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=100,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        verbose=-1,
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        scale_pos_weight=scale_pos,
        random_state=42,
        eval_metric="auc",
    ),
}

In [ ]:
results_fe = []

for model_name, model in models.items():

    print(
        f"Evaluiere Modell: {model_name}..."
    )

    # Ohne Isolation Score
    model.fit(
        X_tr_base,
        y_tr,
    )

    prob_base = model.predict_proba(
        X_va_base
    )[:, 1]

    auc_base = roc_auc_score(
        y_va,
        prob_base,
    )

    # Mit Isolation Score
    model.fit(
        X_tr_aug,
        y_tr,
    )

    prob_aug = model.predict_proba(
        X_va_aug
    )[:, 1]

    auc_aug = roc_auc_score(
        y_va,
        prob_aug,
    )

    results_fe.append({
        "Modell": model_name,
        "ROC-AUC (Baseline)": round(
            auc_base, 4
        ),
        "ROC-AUC (+ Anomaly Score)": round(
            auc_aug, 4
        ),
        "Delta AUC": round(
            auc_aug - auc_base, 4
        ),
        "Normalized Gini (+ Feature)": round(
            2 * auc_aug - 1, 4
        ),
    })

In [ ]:
df_fe_results = pd.DataFrame(
    results_fe
)

df_fe_results

Interpretation: der Isolation Score ist zwar deskriptiv interessant gewesen, verbessert hier die überwachten Modelle aber nicht. Besonders bei LightGBM und XGBoost sinkt die
ROC-AUC sogar leicht, was dafür spricht, dass die Baumverfahren die relevante Risikostruktur bereits aus den ursprünglichen Features ableiten können. Der Isolation Score wird daher später
nicht als zusätzliches Feature übernommen.

### 3.9 Vergleich kategorialer Repräsentationen

Ein weiterer Schwerpunkt aus der weiterführenden EDA war die Frage, wie kategoriale Variablen
für Gradient-Boosting-Modelle dargestellt werden sollten. Verglichen wurden hier konkret One-Hot Encoding, OOF Target Encoding, sowie native Kategorien. Die drei Repräsentationen wurden mit LightGBM, XGBoost und CatBoost ausgetestet.

In [ ]:
feature_cols = [
    col
    for col in non_calc_cols
    if col != TARGET
]

cat_cols = [
    col
    for col in feature_cols
    if col.endswith("_cat")
]

num_cols = [
    col
    for col in feature_cols
    if col not in cat_cols
]

In [ ]:
X_train_raw = train_df[
    feature_cols
].copy()

X_val_raw = validation_df[
    feature_cols
].copy()

y_train = train_df[
    TARGET
].astype("int8")

y_val = validation_df[
    TARGET
].astype("int8")

In [ ]:
scale_pos = (
    y_train.eq(0).sum()
    / y_train.eq(1).sum()
)

scale_pos

In [ ]:
# One-Hot-Encoder definieren mit Fallback
def make_ohe_preprocessor(
    cat_cols,
    num_cols,
    scale_numeric=False,
):

    try:
        one_hot_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )

    except TypeError:
        one_hot_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )

    cat_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                ),
            ),
            (
                "onehot",
                one_hot_encoder,
            ),
        ]
    )

    num_steps = [
        (
            "imputer",
            SimpleImputer(
                strategy="median",
            ),
        ),
    ]

    if scale_numeric:
        num_steps.append(
            (
                "scaler",
                StandardScaler(),
            )
        )

    num_pipeline = Pipeline(
        steps=num_steps
    )

    return ColumnTransformer(
        transformers=[
            (
                "num",
                num_pipeline,
                num_cols,
            ),
            (
                "cat",
                cat_pipeline,
                cat_cols,
            ),
        ]
    )


def make_one_hot_preprocessor(
    cat_cols,
    num_cols,
):
    return make_ohe_preprocessor(
        cat_cols=cat_cols,
        num_cols=num_cols,
        scale_numeric=False,
    )

In [ ]:
ohe_preprocessor = (
    make_one_hot_preprocessor(
        cat_cols=cat_cols,
        num_cols=num_cols,
    )
)

X_train_ohe = (
    ohe_preprocessor.fit_transform(
        X_train_raw
    )
)

X_val_ohe = (
    ohe_preprocessor.transform(
        X_val_raw
    )
)

In [ ]:
# Native Kategorien für LightGBM
def make_native_categorical_frame(
    frame,
    cat_cols,
):
    native_frame = frame.copy()

    for col in cat_cols:
        native_frame[col] = (
            native_frame[col]
            .astype("object")
            .where(
                native_frame[col].notna(),
                "__MISSING__",
            )
            .astype("category")
        )

    return native_frame

In [ ]:
X_train_native = (
    make_native_categorical_frame(
        X_train_raw,
        cat_cols,
    )
)

X_val_native = (
    make_native_categorical_frame(
        X_val_raw,
        cat_cols,
    )
)

In [ ]:
# Native Kategorien für CatBoost
def make_catboost_frame(
    frame,
    cat_cols,
):
    catboost_frame = frame.copy()

    for col in cat_cols:
        catboost_frame[col] = (
            catboost_frame[col]
            .astype("object")
            .where(
                catboost_frame[col].notna(),
                "__MISSING__",
            )
            .astype(str)
        )

    return catboost_frame

In [ ]:
X_train_catboost = (
    make_catboost_frame(
        X_train_raw,
        cat_cols,
    )
)

X_val_catboost = (
    make_catboost_frame(
        X_val_raw,
        cat_cols,
    )
)

In [ ]:
catboost_cat_indices = [
    X_train_catboost.columns.get_loc(
        col
    )
    for col in cat_cols
]

In [ ]:
## Out-of-Fold Target Encoding
def target_encode_oof(
    X_train,
    y_train,
    X_val,
    cat_cols,
    num_cols,
    smoothing=50,
    n_splits=5,
    random_state=42,
):

    y_train = pd.Series(
        y_train,
        index=X_train.index,
    )

    global_mean = y_train.mean()

    X_train_out = pd.DataFrame(
        index=X_train.index
    )

    X_val_out = pd.DataFrame(
        index=X_val.index
    )

    # Numerische Features
    medians = X_train[
        num_cols
    ].median()

    for col in num_cols:
        X_train_out[col] = (
            X_train[col]
            .fillna(medians[col])
            .astype("float32")
        )

        X_val_out[col] = (
            X_val[col]
            .fillna(medians[col])
            .astype("float32")
        )

    folds = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    for col in cat_cols:

        train_col = (
            X_train[col]
            .astype("object")
            .where(
                X_train[col].notna(),
                "__MISSING__",
            )
        )

        val_col = (
            X_val[col]
            .astype("object")
            .where(
                X_val[col].notna(),
                "__MISSING__",
            )
        )

        encoded_train = pd.Series(
            global_mean,
            index=X_train.index,
            dtype="float32",
        )

        for fit_idx, oof_idx in folds.split(
            X_train,
            y_train,
        ):

            fit_keys = train_col.iloc[
                fit_idx
            ]

            fit_target = y_train.iloc[
                fit_idx
            ]

            stats = (
                fit_target
                .groupby(fit_keys)
                .agg(["sum", "count"])
            )

            mapping = (
                (
                    stats["sum"]
                    + smoothing * global_mean
                )
                /
                (
                    stats["count"]
                    + smoothing
                )
            )

            encoded_train.iloc[
                oof_idx
            ] = (
                train_col
                .iloc[oof_idx]
                .map(mapping)
                .fillna(global_mean)
                .astype("float32")
                .to_numpy()
            )

        full_stats = (
            y_train
            .groupby(train_col)
            .agg(["sum", "count"])
        )

        full_mapping = (
            (
                full_stats["sum"]
                + smoothing * global_mean
            )
            /
            (
                full_stats["count"]
                + smoothing
            )
        )

        X_train_out[
            f"{col}_te"
        ] = encoded_train.astype(
            "float32"
        )

        X_val_out[
            f"{col}_te"
        ] = (
            val_col
            .map(full_mapping)
            .fillna(global_mean)
            .astype("float32")
            .to_numpy()
        )

    return (
        X_train_out,
        X_val_out,
    )

In [ ]:
X_train_te, X_val_te = (
    target_encode_oof(
        X_train=X_train_raw,
        y_train=y_train,
        X_val=X_val_raw,
        cat_cols=cat_cols,
        num_cols=num_cols,
        smoothing=50,
        n_splits=5,
    )
)

In [ ]:
# XGBoost native Kategorien-Verarbeitung
# + Sicherstellung, dass Training und Validierung selbe Dtype-Kategorie verwenden
def make_xgb_native_categorical_frame(
    train_frame,
    val_frame,
    cat_cols,
):

    train_out = train_frame.copy()
    val_out = val_frame.copy()

    for col in cat_cols:

        train_values = (
            train_out[col]
            .astype("string")
            .fillna("__MISSING__")
        )

        val_values = (
            val_out[col]
            .astype("string")
            .fillna("__MISSING__")
        )

        categories = list(
            train_values.unique()
        )

        if "__UNKNOWN__" not in categories:
            categories.append(
                "__UNKNOWN__"
            )

        dtype = pd.CategoricalDtype(
            categories=categories
        )

        train_out[col] = (
            train_values.astype(dtype)
        )

        val_values = val_values.where(
            val_values.isin(categories),
            "__UNKNOWN__",
        )

        val_out[col] = (
            val_values.astype(dtype)
        )

    return train_out, val_out

In [ ]:
X_train_native_xgb, X_val_native_xgb = (
    make_xgb_native_categorical_frame(
        X_train_raw,
        X_val_raw,
        cat_cols,
    )
)

In [ ]:
N_ESTIMATORS = 200

# LightGBM Modell definieren
def make_lgb():

    return LGBMClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )

In [ ]:
# XGBoost Modell definieren
def make_xgb(native=False):

    return XGBClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=native,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

In [ ]:
# CatBoost Modell definieren
def make_catboost():

    return CatBoostClassifier(
        iterations=N_ESTIMATORS,
        learning_rate=0.05,
        depth=4,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=RANDOM_STATE,
        allow_writing_files=False,
        verbose=False,
    )

In [ ]:
# Experimente definieren
experiments = [
    {
        "model": "LightGBM",
        "representation": "OHE",
        "estimator": make_lgb(),
        "X_train": X_train_ohe,
        "X_val": X_val_ohe,
    },

    {
        "model": "LightGBM",
        "representation": "OOF Target Encoding",
        "estimator": make_lgb(),
        "X_train": X_train_te,
        "X_val": X_val_te,
    },

    {
        "model": "LightGBM",
        "representation": "Native Kategorien",
        "estimator": make_lgb(),
        "X_train": X_train_native,
        "X_val": X_val_native,
        "fit_kwargs": {
            "categorical_feature": cat_cols,
        },
    },
]

In [ ]:
experiments += [
    {
        "model": "XGBoost",
        "representation": "OHE",
        "estimator": make_xgb(),
        "X_train": X_train_ohe,
        "X_val": X_val_ohe,
    },

    {
        "model": "XGBoost",
        "representation": "OOF Target Encoding",
        "estimator": make_xgb(),
        "X_train": X_train_te,
        "X_val": X_val_te,
    },

    {
        "model": "XGBoost",
        "representation": "Native Kategorien",
        "estimator": make_xgb(
            native=True
        ),
        "X_train": X_train_native_xgb,
        "X_val": X_val_native_xgb,
    },
]

In [ ]:
experiments += [
    {
        "model": "CatBoost",
        "representation": "OHE",
        "estimator": make_catboost(),
        "X_train": X_train_ohe,
        "X_val": X_val_ohe,
    },

    {
        "model": "CatBoost",
        "representation": "OOF Target Encoding",
        "estimator": make_catboost(),
        "X_train": X_train_te,
        "X_val": X_val_te,
    },

    {
        "model": "CatBoost",
        "representation": "Native Kategorien",
        "estimator": make_catboost(),
        "X_train": X_train_catboost,
        "X_val": X_val_catboost,
        "fit_kwargs": {
            "cat_features":
                catboost_cat_indices,
        },
    },
]

In [ ]:
categorical_results = []

for exp in experiments:

    print(
        f'Starte: {exp["model"]} | '
        f'{exp["representation"]}'
    )

    start = time.perf_counter()

    exp["estimator"].fit(
        exp["X_train"],
        y_train,
        **exp.get(
            "fit_kwargs",
            {},
        ),
    )

    runtime = (
        time.perf_counter()
        - start
    )

    proba = (
        exp["estimator"]
        .predict_proba(
            exp["X_val"]
        )[:, 1]
    )

    auc = roc_auc_score(
        y_val,
        proba,
    )

    ap = average_precision_score(
        y_val,
        proba,
    )

    categorical_results.append({
        "Modell":
            exp["model"],

        "Repräsentation":
            exp["representation"],

        "ROC-AUC":
            auc,

        "Normalized Gini":
            2 * auc - 1,

        "Average Precision":
            ap,

        "Trainingszeit (s)":
            runtime,
    })

In [ ]:
categorical_results_df = (
    pd.DataFrame(
        categorical_results
    )
)

categorical_results_df.round({
    "ROC-AUC": 4,
    "Normalized Gini": 4,
    "Average Precision": 4,
    "Trainingszeit (s)": 2,
})

In [ ]:
categorical_results_df.sort_values(
    "ROC-AUC",
    ascending=False,
)

In [ ]:
encoding_pivot = (
    categorical_results_df.pivot(
        index="Repräsentation",
        columns="Modell",
        values="ROC-AUC",
    )
)

encoding_pivot

Interpretation: in diesem Vergleich war XGBoost mit OOF Target Encoding die stärkste
Kombination. OHE war bei XGBoost und LightGBM fast genauso gut. Native Kategorien waren dagegen bei LightGBM und XGBoost schwächer. CatBoost profitierte in diesem Versuch ebenfalls nicht eindeutig von seiner nativen Kategoriebehandlung.

### 3.10 HistGradientBoosting-Kodierungsvergleich mit Kreuzvalidierung

Für HistGradientBoosting wurde die Kodierungsfrage mit vier Repräsentationen später noch einmal separat und mit drei äußeren Kreuzvalidierungs-Folds untersucht.

In [ ]:
OUTER_N_SPLITS = 3
OOF_N_SPLITS = 5
TE_SMOOTHING = 50

In [ ]:
encoding_feature_cols = [
    col
    for col in train_df.columns
    if col not in {
        TARGET,
        "id",
        "ps_ind_09_bin",
    }
    and not col.startswith("ps_calc_")
]

In [ ]:
encoding_cat_cols = [
    col
    for col in encoding_feature_cols
    if col.endswith("_cat")
]

encoding_num_cols = [
    col
    for col in encoding_feature_cols
    if col not in encoding_cat_cols
]

In [ ]:
X_encoding = (
    train_df[
        encoding_feature_cols
    ]
    .copy()
    .reset_index(drop=True)
)

y_encoding = (
    train_df[TARGET]
    .astype("int8")
    .reset_index(drop=True)
)

print(
    "Features:",
    len(encoding_feature_cols),
)

print(
    "Kategorial:",
    len(encoding_cat_cols),
)

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=OUTER_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

encoding_cv_fold_id = np.full(
    len(X_encoding),
    -1,
    dtype=np.int8,
)

In [ ]:
for fold, (_, valid_idx) in enumerate(
    outer_cv.split(
        X_encoding,
        y_encoding,
    )
):
    encoding_cv_fold_id[
        valid_idx
    ] = fold

In [ ]:
pd.Series(
    encoding_cv_fold_id
).value_counts().sort_index()

In [ ]:
def make_ohe_fold(
    X_fit,
    X_valid,
    cat_cols,
    num_cols,
):

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
            ),
        ),
        (
            "onehot",
            encoder,
        ),
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            num_cols,
        ),
        (
            "cat",
            categorical_pipeline,
            cat_cols,
        ),
    ])

    X_fit_out = (
        preprocessor.fit_transform(
            X_fit
        )
    )

    X_valid_out = (
        preprocessor.transform(
            X_valid
        )
    )

    return (
        np.asarray(
            X_fit_out,
            dtype=np.float32,
        ),
        np.asarray(
            X_valid_out,
            dtype=np.float32,
        ),
    )

In [ ]:
def make_native_fold(
    X_fit,
    X_valid,
    cat_cols,
    num_cols,
):

    X_fit_out = X_fit.copy()
    X_valid_out = X_valid.copy()

    medians = X_fit[
        num_cols
    ].median()

    for col in num_cols:

        X_fit_out[col] = (
            pd.to_numeric(
                X_fit_out[col],
                errors="coerce",
            )
            .fillna(medians[col])
            .astype("float32")
        )

        X_valid_out[col] = (
            pd.to_numeric(
                X_valid_out[col],
                errors="coerce",
            )
            .fillna(medians[col])
            .astype("float32")
        )

    for col in cat_cols:

        fit_values = (
            X_fit_out[col]
            .astype("object")
            .where(
                X_fit_out[col].notna(),
                "__MISSING__",
            )
            .astype(str)
        )

        valid_values = (
            X_valid_out[col]
            .astype("object")
            .where(
                X_valid_out[col].notna(),
                "__MISSING__",
            )
            .astype(str)
        )

        categories = list(
            pd.unique(fit_values)
        )

        if "__MISSING__" not in categories:
            categories.append(
                "__MISSING__"
            )

        dtype = pd.CategoricalDtype(
            categories=categories,
            ordered=False,
        )

        valid_values = (
            valid_values.where(
                valid_values.isin(
                    categories
                ),
                "__MISSING__",
            )
        )

        X_fit_out[col] = (
            fit_values.astype(dtype)
        )

        X_valid_out[col] = (
            valid_values.astype(dtype)
        )

    return (
        X_fit_out,
        X_valid_out,
    )

In [ ]:
def target_encode_oof_fold(
    X_fit,
    y_fit,
    X_valid,
    cat_cols,
    num_cols,
    smoothing=50,
    n_splits=5,
    random_state=42,
):

    X_fit = X_fit.copy()
    X_valid = X_valid.copy()

    y_fit_s = pd.Series(
        np.asarray(y_fit),
        index=X_fit.index,
        dtype="float64",
    )

    global_mean = (
        float(y_fit_s.mean())
    )

    X_fit_out = pd.DataFrame(
        index=X_fit.index
    )

    X_valid_out = pd.DataFrame(
        index=X_valid.index
    )

    medians = X_fit[
        num_cols
    ].median()

    for col in num_cols:

        X_fit_out[col] = (
            X_fit[col]
            .fillna(medians[col])
            .astype("float32")
        )

        X_valid_out[col] = (
            X_valid[col]
            .fillna(medians[col])
            .astype("float32")
        )

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    for col in cat_cols:

        fit_keys = (
            X_fit[col]
            .astype("object")
            .where(
                X_fit[col].notna(),
                "__MISSING__",
            )
        )

        valid_keys = (
            X_valid[col]
            .astype("object")
            .where(
                X_valid[col].notna(),
                "__MISSING__",
            )
        )

        encoded_fit = pd.Series(
            global_mean,
            index=X_fit.index,
            dtype=np.float32,
        )

        for inner_fit_pos, inner_valid_pos in (
            inner_cv.split(
                X_fit,
                y_fit_s,
            )
        ):

            inner_df = pd.DataFrame({
                "key": fit_keys.iloc[
                    inner_fit_pos
                ].to_numpy(),

                "target": y_fit_s.iloc[
                    inner_fit_pos
                ].to_numpy(),
            })

            stats = (
                inner_df
                .groupby("key")[
                    "target"
                ]
                .agg(["sum", "count"])
            )

            mapping = (
                (
                    stats["sum"]
                    + smoothing * global_mean
                )
                /
                (
                    stats["count"]
                    + smoothing
                )
            )

            encoded_fit.iloc[
                inner_valid_pos
            ] = (
                fit_keys
                .iloc[inner_valid_pos]
                .map(mapping)
                .fillna(global_mean)
                .astype("float32")
                .to_numpy()
            )

        full_df = pd.DataFrame({
            "key":
                fit_keys.to_numpy(),

            "target":
                y_fit_s.to_numpy(),
        })

        full_stats = (
            full_df
            .groupby("key")[
                "target"
            ]
            .agg(["sum", "count"])
        )

        full_mapping = (
            (
                full_stats["sum"]
                + smoothing * global_mean
            )
            /
            (
                full_stats["count"]
                + smoothing
            )
        )

        X_fit_out[
            f"{col}_te"
        ] = encoded_fit

        X_valid_out[
            f"{col}_te"
        ] = (
            valid_keys
            .map(full_mapping)
            .fillna(global_mean)
            .astype("float32")
            .to_numpy()
        )

    return (
        X_fit_out,
        X_valid_out,
    )

In [ ]:
# Naive Kodierung als Kontrolle
def target_encode_naive_fold(
    X_fit,
    y_fit,
    X_valid,
    cat_cols,
    num_cols,
    smoothing=50,
):

    X_fit = X_fit.copy()
    X_valid = X_valid.copy()

    y_fit_s = pd.Series(
        np.asarray(y_fit),
        index=X_fit.index,
    )

    global_mean = (
        y_fit_s.mean()
    )

    X_fit_out = pd.DataFrame(
        index=X_fit.index
    )

    X_valid_out = pd.DataFrame(
        index=X_valid.index
    )

    medians = X_fit[
        num_cols
    ].median()

    for col in num_cols:

        X_fit_out[col] = (
            X_fit[col]
            .fillna(medians[col])
            .astype("float32")
        )

        X_valid_out[col] = (
            X_valid[col]
            .fillna(medians[col])
            .astype("float32")
        )

    for col in cat_cols:

        fit_keys = (
            X_fit[col]
            .astype("object")
            .where(
                X_fit[col].notna(),
                "__MISSING__",
            )
        )

        valid_keys = (
            X_valid[col]
            .astype("object")
            .where(
                X_valid[col].notna(),
                "__MISSING__",
            )
        )

        temp = pd.DataFrame({
            "key": fit_keys,
            "target": y_fit_s,
        })

        stats = (
            temp
            .groupby("key")[
                "target"
            ]
            .agg(["sum", "count"])
        )

        mapping = (
            (
                stats["sum"]
                + smoothing * global_mean
            )
            /
            (
                stats["count"]
                + smoothing
            )
        )

        X_fit_out[
            f"{col}_te"
        ] = (
            fit_keys
            .map(mapping)
            .fillna(global_mean)
            .astype("float32")
        )

        X_valid_out[
            f"{col}_te"
        ] = (
            valid_keys
            .map(mapping)
            .fillna(global_mean)
            .astype("float32")
        )

    return (
        X_fit_out,
        X_valid_out,
    )

In [ ]:
HGB_ENCODING_BASELINE = {
    "loss": "log_loss",

    "learning_rate": 0.10,
    "max_iter": 2000,

    "max_leaf_nodes": 31,
    "max_depth": None,
    "min_samples_leaf": 20,

    "l2_regularization": 0.0,
    "max_features": 1.0,
    "max_bins": 255,

    "early_stopping": True,
    "validation_fraction": 0.10,
    "n_iter_no_change": 10,
    "tol": 1e-7,

    "random_state": 42,
}

In [ ]:
representations = [
    "Native Kategorien",
    "One-Hot Encoding",
    "OOF Target Encoding",
    "Naives Target Encoding",
]

fold_results = []

In [ ]:
for fold in range(
    OUTER_N_SPLITS
):

    print(
        f"Fold {fold + 1}"
        f"/{OUTER_N_SPLITS}"
    )

    valid_pos = np.flatnonzero(
        encoding_cv_fold_id == fold
    )

    fit_pos = np.flatnonzero(
        encoding_cv_fold_id != fold
    )

    X_fit_raw = (
        X_encoding
        .iloc[fit_pos]
        .copy()
        .reset_index(drop=True)
    )

    X_valid_raw = (
        X_encoding
        .iloc[valid_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_fit = (
        y_encoding
        .iloc[fit_pos]
        .to_numpy()
    )

    y_valid = (
        y_encoding
        .iloc[valid_pos]
        .to_numpy()
    )

    for representation in (
        representations
    ):

        print(representation)

        if representation == (
            "Native Kategorien"
        ):

            X_fit_model, X_valid_model = (
                make_native_fold(
                    X_fit_raw,
                    X_valid_raw,
                    encoding_cat_cols,
                    encoding_num_cols,
                )
            )

            categorical_features = (
                encoding_cat_cols
            )

        elif representation == (
            "One-Hot Encoding"
        ):

            X_fit_model, X_valid_model = (
                make_ohe_fold(
                    X_fit_raw,
                    X_valid_raw,
                    encoding_cat_cols,
                    encoding_num_cols,
                )
            )

            categorical_features = None

        elif representation == (
            "OOF Target Encoding"
        ):

            X_fit_model, X_valid_model = (
                target_encode_oof_fold(
                    X_fit=X_fit_raw,
                    y_fit=y_fit,
                    X_valid=X_valid_raw,
                    cat_cols=encoding_cat_cols,
                    num_cols=encoding_num_cols,
                    smoothing=TE_SMOOTHING,
                    n_splits=OOF_N_SPLITS,
                    random_state=RANDOM_STATE,
                )
            )

            categorical_features = None

        else:

            X_fit_model, X_valid_model = (
                target_encode_naive_fold(
                    X_fit=X_fit_raw,
                    y_fit=y_fit,
                    X_valid=X_valid_raw,
                    cat_cols=encoding_cat_cols,
                    num_cols=encoding_num_cols,
                    smoothing=TE_SMOOTHING,
                )
            )

            categorical_features = None

        model_params = dict(
            HGB_ENCODING_BASELINE
        )

        model_params[
            "categorical_features"
        ] = categorical_features

        model = (
            HistGradientBoostingClassifier(
                **model_params
            )
        )

        model.fit(
            X_fit_model,
            y_fit,
        )

        valid_probability = (
            model.predict_proba(
                X_valid_model
            )[:, 1]
        )

        valid_auc = roc_auc_score(
            y_valid,
            valid_probability,
        )

        valid_ap = (
            average_precision_score(
                y_valid,
                valid_probability,
            )
        )

        fold_results.append({
            "representation":
                representation,

            "fold":
                fold,

            "roc_auc":
                valid_auc,

            "average_precision":
                valid_ap,

            "n_model_features":
                X_fit_model.shape[1],

            "n_iter":
                model.n_iter_,
        })

        print(
            f"ROC-AUC={valid_auc:.4f}"
        )

In [ ]:
encoding_fold_results = (
    pd.DataFrame(
        fold_results
    )
)

encoding_fold_results

In [ ]:
encoding_summary = (
    encoding_fold_results
    .groupby(
        "representation",
        as_index=False,
    )
    .agg(
        roc_auc_mean=(
            "roc_auc",
            "mean",
        ),
        roc_auc_std=(
            "roc_auc",
            "std",
        ),
        average_precision_mean=(
            "average_precision",
            "mean",
        ),
        average_precision_std=(
            "average_precision",
            "std",
        ),
    )
)

encoding_summary

In [ ]:
# für Übersichtsplot
hgb_encoding_cv = pd.DataFrame([
    {
        "Repräsentation": "Native Kategorien",
        "CV ROC-AUC Mittel": 0.621207,
        "CV ROC-AUC Std": 0.001769,
    },
    {
        "Repräsentation": "One-Hot Encoding",
        "CV ROC-AUC Mittel": 0.634067,
        "CV ROC-AUC Std": 0.002546,
    },
    {
        "Repräsentation": "OOF Target Encoding",
        "CV ROC-AUC Mittel": 0.635053,
        "CV ROC-AUC Std": 0.003223,
    },
    {
        "Repräsentation": "Naives Target Encoding",
        "CV ROC-AUC Mittel": 0.634013,
        "CV ROC-AUC Std": 0.002069,
    },
])

hgb_encoding_cv.sort_values(
    "CV ROC-AUC Mittel",
    ascending=False,
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

plot_data = hgb_encoding_cv.sort_values(
    "CV ROC-AUC Mittel"
)

ax.barh(
    plot_data["Repräsentation"],
    plot_data["CV ROC-AUC Mittel"],
    xerr=plot_data["CV ROC-AUC Std"],
    capsize=3,
)

ax.set(
    title="HGB: Vergleich der Feature-Repräsentationen",
    xlabel="mittlere CV ROC-AUC",
    ylabel="",
)

plt.tight_layout()
plt.show()

Out-of-Fold Target Encoding erzielte im Mittel die höchste ROC-AUC. Der Abstand zu OHE
ist aber klein. Native Kategorien fielen in diesem Experiment deutlicher zurück. Das Ergebnis spricht dafür, die Kodierung nicht nur nach theoretischen Erwartungen auszuwählen, sondern auch als Modellierungsentscheidung zu behandeln.

### 3.11 Konsolidierter Modellvergleich

Nach den einzelnen Vorversuchen wurden mehrere Modellfamilien noch einmal unter
einer jeweils sinnvollen Feature-Repräsentation verglichen.

Dieser Vergleich ist noch kein finales HPO. Er dient eher dazu, die Modelle zu
identifizieren, für die sich ein größerer Tuning-Aufwand lohnt.

In [ ]:
drop_cols = {
    TARGET,
    "id",
}

ps_calc_cols = [
    col
    for col in train_df.columns
    if col.startswith("ps_calc_")
]

redundant_cols = [
    "ps_ind_09_bin"
]

In [ ]:
final_feature_cols = [
    col
    for col in train_df.columns
    if col not in drop_cols
    and col not in ps_calc_cols
    and col not in redundant_cols
]

len(final_feature_cols)

In [ ]:
cat_cols = [
    col
    for col in final_feature_cols
    if col.endswith("_cat")
]

num_cols = [
    col
    for col in final_feature_cols
    if col not in cat_cols
]

In [ ]:
X_train_raw = train_df[
    final_feature_cols
].copy()

X_val_raw = validation_df[
    final_feature_cols
].copy()

y_train = train_df[
    TARGET
].astype("int8")

y_val = validation_df[
    TARGET
].astype("int8")

In [ ]:
scale_pos_weight = (
    y_train.eq(0).sum()
    / y_train.eq(1).sum()
)

In [ ]:
ohe_tree_preprocessor = (
    make_ohe_preprocessor(
        cat_cols=cat_cols,
        num_cols=num_cols,
        scale_numeric=False,
    )
)

X_train_ohe_tree = (
    ohe_tree_preprocessor
    .fit_transform(
        X_train_raw
    )
)

X_val_ohe_tree = (
    ohe_tree_preprocessor
    .transform(
        X_val_raw
    )
)

In [ ]:
X_train_hgb_ohe, X_val_hgb_ohe = (
    make_ohe_fold(
        X_fit=X_train_raw,
        X_valid=X_val_raw,
        cat_cols=cat_cols,
        num_cols=num_cols,
    )
)

In [ ]:
X_train_hgb_native, X_val_hgb_native = (
    make_native_fold(
        X_fit=X_train_raw,
        X_valid=X_val_raw,
        cat_cols=cat_cols,
        num_cols=num_cols,
    )
)

In [ ]:
ohe_scaled_preprocessor = (
    make_ohe_preprocessor(
        cat_cols=cat_cols,
        num_cols=num_cols,
        scale_numeric=True,
    )
)

X_train_ohe_scaled = (
    ohe_scaled_preprocessor
    .fit_transform(
        X_train_raw
    )
)

X_val_ohe_scaled = (
    ohe_scaled_preprocessor
    .transform(
        X_val_raw
    )
)

In [ ]:
X_train_native_lgb = (
    make_native_categorical_frame(
        X_train_raw,
        cat_cols,
    )
)

X_val_native_lgb = (
    make_native_categorical_frame(
        X_val_raw,
        cat_cols,
    )
)

In [ ]:
X_train_catboost = (
    make_catboost_frame(
        X_train_raw,
        cat_cols,
    )
)

X_val_catboost = (
    make_catboost_frame(
        X_val_raw,
        cat_cols,
    )
)

catboost_cat_indices = [
    X_train_catboost.columns.get_loc(
        col
    )
    for col in cat_cols
]

In [ ]:
X_train_te, X_val_te = (
    target_encode_oof(
        X_train=X_train_raw,
        y_train=y_train,
        X_val=X_val_raw,
        cat_cols=cat_cols,
        num_cols=num_cols,
        smoothing=50,
        n_splits=5,
    )
)

In [ ]:
def evaluate_probability_model(
    name,
    model,
    X_train,
    X_val,
    y_train,
    y_val,
    fit_kwargs=None,
):

    fit_kwargs = (
        fit_kwargs or {}
    )

    start = time.perf_counter()

    model.fit(
        X_train,
        y_train,
        **fit_kwargs,
    )

    runtime = (
        time.perf_counter()
        - start
    )

    proba = model.predict_proba(
        X_val
    )[:, 1]

    auc = roc_auc_score(
        y_val,
        proba,
    )

    ap = average_precision_score(
        y_val,
        proba,
    )

    y_val_array = np.asarray(
        y_val
    )

    n_top = int(
        np.ceil(
            len(y_val_array) * 0.10
        )
    )

    top_idx = np.argsort(
        proba
    )[::-1][:n_top]

    base_rate = (
        y_val_array.mean()
    )

    top_rate = (
        y_val_array[
            top_idx
        ].mean()
    )

    recall_top10 = (
        y_val_array[
            top_idx
        ].sum()
        / y_val_array.sum()
    )

    lift_top10 = (
        top_rate
        / base_rate
    )

    return {
        "Modell": name,

        "ROC-AUC": auc,

        "Normalized Gini":
            2 * auc - 1,

        "Average Precision":
            ap,

        "Top-10%-Schadenquote":
            top_rate,

        "Recall@Top10%":
            recall_top10,

        "Lift@Top10%":
            lift_top10,

        "Trainingszeit (s)":
            runtime,
    }

In [ ]:
N_ESTIMATORS = 200
model_specs = []

In [ ]:
model_specs += [
    {
        "name": "Dummy Prior",

        "model":
            DummyClassifier(
                strategy="prior"
            ),

        "X_train":
            X_train_ohe_tree,

        "X_val":
            X_val_ohe_tree,
    },

    {
        "name":
            "Logistische Regression",

        "model":
            LogisticRegression(
                max_iter=1500,
                solver="saga",
                l1_ratio=0,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),

        "X_train":
            X_train_ohe_scaled,

        "X_val":
            X_val_ohe_scaled,
    },
]

In [ ]:
model_specs.append({
    "name": "Decision Tree",

    "model": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),

    "X_train":
        X_train_ohe_tree,

    "X_val":
        X_val_ohe_tree,
})

In [ ]:
model_specs.append({
    "name": "Random Forest",

    "model": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=100,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "X_train":
        X_train_ohe_tree,

    "X_val":
        X_val_ohe_tree,
})

In [ ]:
model_specs.append({
    "name": "MLP",

    "model": MLPClassifier(
        hidden_layer_sizes=(
            64,
            32,
        ),
        max_iter=80,
        early_stopping=True,
        random_state=RANDOM_STATE,
    ),

    "X_train":
        X_train_ohe_scaled,

    "X_val":
        X_val_ohe_scaled,
})

In [ ]:
model_specs.append({
    "name": "LightGBM + OHE",

    "model": LGBMClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),

    "X_train":
        X_train_ohe_tree,

    "X_val":
        X_val_ohe_tree,
})

In [ ]:
model_specs.append({
    "name":
        "LightGBM native Kategorien",

    "model": LGBMClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),

    "X_train":
        X_train_native_lgb,

    "X_val":
        X_val_native_lgb,

    "fit_kwargs": {
        "categorical_feature":
            cat_cols,
    },
})

In [ ]:
model_specs.append({
    "name":
        "LightGBM + OOF Target Encoding",

    "model":
        LGBMClassifier(
            n_estimators=N_ESTIMATORS,
            learning_rate=0.05,
            max_depth=4,
            num_leaves=15,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ),

    "X_train":
        X_train_te,

    "X_val":
        X_val_te,
})

In [ ]:
model_specs.append({
    "name":
        "HistGradientBoosting + OHE",

    "model":
        HistGradientBoostingClassifier(
            **HGB_ENCODING_BASELINE
        ),

    "X_train":
        X_train_hgb_ohe,

    "X_val":
        X_val_hgb_ohe,
})

In [ ]:
model_specs.append({
    "name":
        "HistGradientBoosting + OOF Target Encoding",

    "model":
        HistGradientBoostingClassifier(
            **HGB_ENCODING_BASELINE
        ),

    "X_train":
        X_train_te,

    "X_val":
        X_val_te,
})

In [ ]:
hgb_native_params = dict(
    HGB_ENCODING_BASELINE
)

hgb_native_params[
    "categorical_features"
] = cat_cols

In [ ]:
model_specs.append({
    "name":
        "HistGradientBoosting native Kategorien",

    "model":
        HistGradientBoostingClassifier(
            **hgb_native_params
        ),

    "X_train":
        X_train_hgb_native,

    "X_val":
        X_val_hgb_native,
})

In [ ]:
model_specs.append({
    "name": "XGBoost + OHE",

    "model": XGBClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=
            scale_pos_weight,
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "X_train":
        X_train_ohe_tree,

    "X_val":
        X_val_ohe_tree,
})

In [ ]:
model_specs.append({
    "name":
        "XGBoost + OOF Target Encoding",

    "model": XGBClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=
            scale_pos_weight,
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "X_train":
        X_train_te,

    "X_val":
        X_val_te,
})

In [ ]:
def make_xgb_native_categorical_frame(
    train_frame,
    val_frame,
    cat_cols,
):
    train_out = train_frame.copy()
    val_out = val_frame.copy()

    for col in cat_cols:

        train_values = (
            train_out[col]
            .astype("string")
            .fillna("__MISSING__")
        )

        val_values = (
            val_out[col]
            .astype("string")
            .fillna("__MISSING__")
        )

        categories = list(
            train_values.unique()
        )

        if "__UNKNOWN__" not in categories:
            categories.append(
                "__UNKNOWN__"
            )

        dtype = pd.CategoricalDtype(
            categories=categories
        )

        train_out[col] = (
            train_values.astype(dtype)
        )

        val_values = val_values.where(
            val_values.isin(categories),
            "__UNKNOWN__",
        )

        val_out[col] = (
            val_values.astype(dtype)
        )

    return train_out, val_out

In [ ]:
X_train_native_xgb, X_val_native_xgb = (
    make_xgb_native_categorical_frame(
        X_train_raw,
        X_val_raw,
        cat_cols,
    )
)

In [ ]:
model_specs.append({
    "name":
        "XGBoost native Kategorien",

    "model":
        XGBClassifier(
            n_estimators=N_ESTIMATORS,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=
                scale_pos_weight,
            eval_metric="auc",
            tree_method="hist",
            enable_categorical=True,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),

    "X_train":
        X_train_native_xgb,

    "X_val":
        X_val_native_xgb,
})

In [ ]:
model_specs.append({
    "name":
        "CatBoost native Kategorien",

    "model": CatBoostClassifier(
        iterations=N_ESTIMATORS,
        learning_rate=0.05,
        depth=4,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights=
            "Balanced",
        random_seed=RANDOM_STATE,
        allow_writing_files=False,
        verbose=False,
    ),

    "X_train":
        X_train_catboost,

    "X_val":
        X_val_catboost,

    "fit_kwargs": {
        "cat_features":
            catboost_cat_indices,
    },
})

In [ ]:
model_specs.append({
    "name":
        "CatBoost + OHE",

    "model":
        CatBoostClassifier(
            iterations=N_ESTIMATORS,
            learning_rate=0.05,
            depth=4,
            loss_function="Logloss",
            eval_metric="AUC",
            auto_class_weights=
                "Balanced",
            random_seed=RANDOM_STATE,
            allow_writing_files=False,
            verbose=False,
        ),

    "X_train":
        X_train_ohe_tree,

    "X_val":
        X_val_ohe_tree,
})

In [ ]:
model_specs.append({
    "name":
        "CatBoost + OOF Target Encoding",

    "model":
        CatBoostClassifier(
            iterations=N_ESTIMATORS,
            learning_rate=0.05,
            depth=4,
            loss_function="Logloss",
            eval_metric="AUC",
            auto_class_weights=
                "Balanced",
            random_seed=RANDOM_STATE,
            allow_writing_files=False,
            verbose=False,
        ),

    "X_train":
        X_train_te,

    "X_val":
        X_val_te,
})

In [ ]:
screening_results = []

for spec in model_specs:

    print(
        f"Starte: "
        f"{spec['name']}"
    )

    result = (
        evaluate_probability_model(
            name=spec["name"],
            model=spec["model"],
            X_train=spec["X_train"],
            X_val=spec["X_val"],
            y_train=y_train,
            y_val=y_val,
            fit_kwargs=spec.get(
                "fit_kwargs",
                {},
            ),
        )
    )

    screening_results.append(
        result
    )

    print(
        f"ROC-AUC="
        f"{result['ROC-AUC']:.4f}"
    )

In [ ]:
consolidated_benchmark = (
    pd.DataFrame(
        screening_results
    )
    .sort_values(
        "ROC-AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)

consolidated_benchmark

In [ ]:
boosting_comparison = (
    consolidated_benchmark[
        consolidated_benchmark[
            "Modell"
        ].str.contains(
            "LightGBM|HistGradientBoosting|XGBoost|CatBoost"
        )
    ]
    .copy()
)

Die drei stärksten Baselines sind damit XGBoost, HistGradientBoosting und LightGBM. Der Abstand zu linearen Modellen, Random Forest und MLP ist zwar nicht riesig, aber konsistent. Für die späteren HPO- und Ensembleversuche liegt der Schwerpunkt deshalb auf
Boosting-Entscheidungsbaum-Ensemble-Methoden.

### 3.12 Missing-Value-Strategien im direkten Vergleich

Rund 79% der Zeilen enthalten mindestens einen fehlenden Wert. Ein vollständiges Entfernen würde den Datensatz drastisch und nicht akzeptabel dezimieren. Daher wurden im Folgenden verschiedene Strategien verglichen: Median-Imputation mit Missing-Indikatoren, native NaN-Verarbeitung, sowie, nur als Vergleich das vollständige Fehlwert-Entfernen.

In [ ]:
# Falls vorhanden GPU für XGBoost nehmen, sonst CPU
try:
    import torch

    XGB_DEVICE = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

except ImportError:
    XGB_DEVICE = "cpu"

print(
    "XGBoost Device:",
    XGB_DEVICE,
)

In [ ]:
calc_cols = [
    col
    for col in train_df.columns
    if col.startswith("ps_calc_")
]

non_calc_cols = [
    col
    for col in train_df.columns
    if col not in (
        set(calc_cols)
        | {
            "id",
            TARGET,
            "ps_ind_09_bin",
        }
    )
]

print(
    "Anzahl Features:",
    len(non_calc_cols),
)

In [ ]:
X_train_missing = (
    train_df[
        non_calc_cols
    ]
    .copy()
    .replace(
        -1,
        np.nan,
    )
    .reset_index(
        drop=True
    )
)

X_val_missing = (
    validation_df[
        non_calc_cols
    ]
    .copy()
    .replace(
        -1,
        np.nan,
    )
    .reset_index(
        drop=True
    )
)

In [ ]:
y_train_missing = (
    train_df[TARGET]
    .astype("int8")
    .reset_index(drop=True)
)

y_val_missing = (
    validation_df[TARGET]
    .astype("int8")
    .reset_index(drop=True)
)

In [ ]:
print(
    "Train:",
    X_train_missing.shape,
)

print(
    "Validation:",
    X_val_missing.shape,
)

print(
    "Features mit Missing Values:",
    X_train_missing
    .isna()
    .any()
    .sum(),
)

print(
    "Zeilen mit Missing Values:",
    f"{X_train_missing.isna().any(axis=1).mean():.2%}",
)

In [ ]:
missing_profile = pd.DataFrame({
    "Feature":
        X_train_missing.columns,

    "Missing-Anteil Training":
        X_train_missing
        .isna()
        .mean()
        .values,

    "Missing-Anteil Validation":
        X_val_missing
        .isna()
        .mean()
        .values,
})

In [ ]:
missing_profile = (
    missing_profile[
        missing_profile[
            "Missing-Anteil Training"
        ] > 0
    ]
    .sort_values(
        "Missing-Anteil Training",
        ascending=False,
    )
    .reset_index(drop=True)
)

missing_profile.round(4)

In [ ]:
def prepare_missing_strategy(
    strategy,
    X_train,
    X_val,
    y_train,
    y_val,
):

    X_tr = X_train.copy()
    X_va = X_val.copy()

    y_tr = y_train.copy()
    y_va = y_val.copy()

    start = time.perf_counter()

    if strategy == "median_indicator":

        imputer = SimpleImputer(
            strategy="median",
            add_indicator=True,
        )

        X_tr_out = (
            imputer.fit_transform(
                X_tr
            )
        )

        X_va_out = (
            imputer.transform(
                X_va
            )
        )

        X_tr_out = (
            X_tr_out.astype(
                "float32"
            )
        )

        X_va_out = (
            X_va_out.astype(
                "float32"
            )
        )

    elif strategy == "native_nan":

        X_tr_out = (
            X_tr.astype(
                "float32"
            )
        )

        X_va_out = (
            X_va.astype(
                "float32"
            )
        )

    elif strategy == "complete_case":

        train_mask = (
            ~X_tr
            .isna()
            .any(axis=1)
        )

        val_mask = (
            ~X_va
            .isna()
            .any(axis=1)
        )

        X_tr_out = (
            X_tr.loc[
                train_mask
            ]
            .astype("float32")
            .reset_index(
                drop=True
            )
        )

        X_va_out = (
            X_va.loc[
                val_mask
            ]
            .astype("float32")
            .reset_index(
                drop=True
            )
        )

        y_tr = (
            y_tr.loc[
                train_mask
            ]
            .reset_index(
                drop=True
            )
        )

        y_va = (
            y_va.loc[
                val_mask
            ]
            .reset_index(
                drop=True
            )
        )

    else:
        raise ValueError(
            f"Unbekannte Strategie: {strategy}"
        )

    preprocessing_time = (
        time.perf_counter()
        - start
    )

    return {
        "X_train":
            X_tr_out,

        "X_val":
            X_va_out,

        "y_train":
            y_tr,

        "y_val":
            y_va,

        "train_retention":
            len(y_tr)
            / len(y_train),

        "val_retention":
            len(y_va)
            / len(y_val),

        "n_features":
            X_tr_out.shape[1],

        "preprocessing_time":
            preprocessing_time,
    }

In [ ]:
def make_missing_models():

    models = {}

    models[
        "Logistische Regression"
    ] = Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1500,
                solver="saga",
                random_state=
                    RANDOM_STATE,
            ),
        ),
    ])

    models[
        "Random Forest"
    ] = RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        min_samples_leaf=100,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    models[
        "HistGradientBoosting"
    ] = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_depth=4,
        max_leaf_nodes=15,
        early_stopping=False,
        random_state=RANDOM_STATE,
    )

    models[
        "XGBoost"
    ] = XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=1.0,
        colsample_bytree=1.0,
        tree_method="hist",
        device=XGB_DEVICE,
        missing=np.nan,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    return models

In [ ]:
def evaluate_missing_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
):

    sample_weight = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train,
        )
    )

    if isinstance(
        model,
        Pipeline,
    ):

        fit_kwargs = {
            "model__sample_weight":
                sample_weight,
        }

    else:

        fit_kwargs = {
            "sample_weight":
                sample_weight,
        }

    start = (
        time.perf_counter()
    )

    model.fit(
        X_train,
        y_train,
        **fit_kwargs,
    )

    fit_time = (
        time.perf_counter()
        - start
    )

    start = (
        time.perf_counter()
    )

    probabilities = (
        model
        .predict_proba(
            X_val
        )[:, 1]
    )

    inference_time = (
        time.perf_counter()
        - start
    )

    auc = roc_auc_score(
        y_val,
        probabilities,
    )

    ap = (
        average_precision_score(
            y_val,
            probabilities,
        )
    )

    ll = log_loss(
        y_val,
        probabilities,
    )

    y_val_array = (
        np.asarray(
            y_val
        )
    )

    n_top = int(
        np.ceil(
            len(y_val_array)
            * 0.10
        )
    )

    top_idx = (
        np.argsort(
            probabilities
        )[::-1][:n_top]
    )

    base_claim_rate = (
        y_val_array.mean()
    )

    top_claim_rate = (
        y_val_array[
            top_idx
        ].mean()
    )

    recall_top10 = (
        y_val_array[
            top_idx
        ].sum()
        / y_val_array.sum()
    )

    lift_top10 = (
        top_claim_rate
        / base_claim_rate
    )

    return {
        "ROC-AUC":
            auc,

        "Normalized Gini":
            2 * auc - 1,

        "Average Precision":
            ap,

        "Log Loss":
            ll,

        "Top-10%-Schadenquote":
            top_claim_rate,

        "Recall@Top10%":
            recall_top10,

        "Lift@Top10%":
            lift_top10,

        "Modell-Fit-Zeit (s)":
            fit_time,

        "Inferenzzeit (s)":
            inference_time,
    }

In [ ]:
strategies = {
    "Median + Missing-Indikatoren":
        "median_indicator",

    "Native NaN-Verarbeitung":
        "native_nan",

    "Vollständiges Entfernen":
        "complete_case",
}

In [ ]:
native_nan_models = {
    "HistGradientBoosting",
    "XGBoost",
}

In [ ]:
missing_results = []

for (
    strategy_label,
    strategy_key,
) in strategies.items():

    print()
    print("=" * 75)

    print(
        "Missing-Strategie:",
        strategy_label,
    )

    print("=" * 75)

    scenario = (
        prepare_missing_strategy(
            strategy=
                strategy_key,

            X_train=
                X_train_missing,

            X_val=
                X_val_missing,

            y_train=
                y_train_missing,

            y_val=
                y_val_missing,
        )
    )

    X_tr = scenario[
        "X_train"
    ]

    X_va = scenario[
        "X_val"
    ]

    y_tr = scenario[
        "y_train"
    ]

    y_va = scenario[
        "y_val"
    ]

    print(
        "Training behalten:",
        f"{scenario['train_retention']:.2%}",
    )

    print(
        "Validation behalten:",
        f"{scenario['val_retention']:.2%}",
    )

    print(
        "Anzahl Features:",
        scenario[
            "n_features"
        ],
    )

    models = (
        make_missing_models()
    )

    for (
        model_name,
        model,
    ) in models.items():

        if (
            strategy_key
            == "native_nan"
            and model_name
            not in native_nan_models
        ):
            continue

        print(
            f"\nStarte: "
            f"{model_name}"
        )

        metrics = (
            evaluate_missing_model(
                model=model,

                X_train=X_tr,
                y_train=y_tr,

                X_val=X_va,
                y_val=y_va,
            )
        )

        end_to_end_time = (
            scenario[
                "preprocessing_time"
            ]
            + metrics[
                "Modell-Fit-Zeit (s)"
            ]
            + metrics[
                "Inferenzzeit (s)"
            ]
        )

        missing_results.append({
            "Modell":
                model_name,

            "Missing-Strategie":
                strategy_label,

            **metrics,

            "Train-Retention":
                scenario[
                    "train_retention"
                ],

            "Validation-Retention":
                scenario[
                    "val_retention"
                ],

            "Anzahl Features":
                scenario[
                    "n_features"
                ],

            "Preprocessing-Zeit (s)":
                scenario[
                    "preprocessing_time"
                ],

            "End-to-End-Zeit (s)":
                end_to_end_time,
        })

        print(
            f"ROC-AUC="
            f"{metrics['ROC-AUC']:.4f}"
            f" | AP="
            f"{metrics['Average Precision']:.4f}"
            f" | Fit="
            f"{metrics['Modell-Fit-Zeit (s)']:.2f}s"
        )

In [ ]:
missing_benchmark = (
    pd.DataFrame(
        missing_results
    )
    .sort_values(
        [
            "Modell",
            "ROC-AUC",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

In [ ]:
missing_benchmark.round({
    "ROC-AUC": 4,
    "Normalized Gini": 4,
    "Average Precision": 4,
    "Log Loss": 4,

    "Top-10%-Schadenquote": 4,
    "Recall@Top10%": 4,
    "Lift@Top10%": 4,

    "Train-Retention": 4,
    "Validation-Retention": 4,

    "Preprocessing-Zeit (s)": 2,
    "Modell-Fit-Zeit (s)": 2,
    "Inferenzzeit (s)": 2,
    "End-to-End-Zeit (s)": 2,
})

In [ ]:
roc_missing_pivot = (
    missing_benchmark.pivot(
        index="Modell",
        columns="Missing-Strategie",
        values="ROC-AUC",
    )
)

roc_missing_pivot.round(4)

In [ ]:
ap_missing_pivot = (
    missing_benchmark.pivot(
        index="Modell",
        columns="Missing-Strategie",
        values="Average Precision",
    )
)

ap_missing_pivot.round(4)

Für HGB und XGBoost ist die native Missing-Value-Behandlung am stärksten.
Complete-Case reduziert die Trainingsmenge auf ungefähr 21 % und verschlechtert
die ROC-AUC der Boosting-Modelle deutlich.

Die zum Teil höhere Average Precision im Complete-Case-Datensatz ist nicht
direkt mit der vollständigen Validation vergleichbar, weil sich die evaluierte
Teilpopulation verändert.

## 4. Zwischenfazit der EDA

Aus den EDA- und ersten Modell-Experimenten ergeben sich mehrere Erkenntnisse und, teilweise auch in der Gruppe zusammen entschiedene, Arbeitsentscheidungen: 

1. Wie sich gezeigt hat ist die Klassenverteilung stark unausgeglichen. ROC-AUC bzw. der normalisierte Gini-Koeffizient scheinen deshalb wichtiger als die Akkuratheit als Metrik.
2. Missing Values werden nicht einfach vollständig entfernt. Bei Boosting-Modellen ist die
   native Missing-Value-Verarbeitung besonders interessant.
3. Die ps_calc_*-Featurefamilie liefert in den bisherigen Experimenten kaum zusätzliches Zielvariablen-Signal. In der Gruppe wurde deswegen entschieden diese Featurefamilie für das Hyperparameter-Tuning zu entfernen.
4. Die binäre Gruppe ps_ind_06_bin bis ps_ind_09_bin ergibt für jede Trainingszeile = 1, d.h.  ein Feature kann aus den restlichen rekonstruiert werden.
5. Preprocessing wirkt je nach Modellfamilie unterschiedlich stark.
6. Der Isolation-Forest-Score beschreibt zwar Risikostruktur, bringt als
   zusätzliches Feature aber keinen Vorteil für die Entscheidungsbaum-Ensembles.
7. OHE und OOF Target Encoding sind für die stärkeren Boosting-Modelle deutlich
   konkurrenzfähiger als die getesteten nativen Kategorievarianten.
8. HistGradientBoosting, XGBoost und LightGBM u.a. sind die stärksten Baseline-Familien und werden deshalb für die Hyperparameter-Optimierung und Ensembles priorisiert. Die logistische Regression bleibt die Baseline.
9. Ein vollständiges Entfernen von Beobachtungen mit Fehlwerten ist wegen des großen Datenverlusts keine sinnvolle    Standardstrategie.

## 5. Hyperparameteroptimierung (HPO) und Trainingsprozess

Bei der Besprechung für die nachfolgende detailliertere Modell-Untersuchung mit Hyperparameter-Suche etc. wurde mir in der Gruppe die der HistGradientBoostingClassifier zugewiesen. Bei der weiteren Nutzung wurden, basierend auf den bisherigen Erkenntnissen die ps_calc_*-Features weggelassen und ps_ind_09_bin als Referenzvariable entfernt. Das ursprüngliche Hyperparameter-Optimierungsnotebook enthielt zusätzlich Checkpoints und Wiederaufnahmelogik im Falle eines Absturzes, welche hier der Übersichtlichkeit halber nicht übernommen wurden.

### 5.1 Features und Kreuzvalidierung definieren

In [ ]:
from sklearn.model_selection import StratifiedKFold

hpo_features = [
    col for col in train_df.columns
    if col not in {"id", TARGET, "ps_ind_09_bin"}
    and not col.startswith("ps_calc_")
]

hpo_cat_features = [
    col for col in hpo_features
    if col.endswith("_cat")
]

hpo_num_features = [
    col for col in hpo_features
    if col not in hpo_cat_features
]

print("Features:", len(hpo_features))
print("Kategorial:", len(hpo_cat_features))
print("Numerisch / binär:", len(hpo_num_features))

In [ ]:
hpo_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)

X_hpo = train_df[hpo_features].copy()
y_hpo = train_df[TARGET].to_numpy()

### 5.2 HistGradientBoosting

Es wurden mehrere Suchstrategien getestet, Random Search als Baseline, sowie NSGA-II und TPE als adaptive Algorithmen. Es wurden jeweils 200 Trials durchgeführt. Für eine übersichtlichere Notebook-Version wird hier beipsielhaft die Pipeline mit Random Search aufgezeigt, abgesehen von den algorithmus-spezifischen Unterschieden war sie jedes mal gleich.

In [ ]:
HGB_SEARCH_SPACE = {
    "learning_rate": {
        "low": 0.003,
        "high": 0.3,
    },

    "tree_complexity_mode": [
        "leaf_controlled",
        "depth_controlled",
    ],

    "max_leaf_nodes": {
        "low": 3,
        "high": 255,
    },

    "max_depth": {
        "low": 2,
        "high": 16,
    },

    "min_samples_leaf": {
        "low": 5,
        "high": 5000,
    },

    "l2_regularization": {
        "zero_probability": 0.10,
        "low_nonzero": 1e-8,
        "high_nonzero": 1000.0,
    },

    "max_features": {
        "low": 0.2,
        "high": 1.0,
    },

    "max_bins": {
        "low": 104,
        "high": 255,
    },

    "interaction_cst": [
        None,
        "pairwise",
        "no_interactions",
    ],

    "validation_fraction": {
        "low": 0.05,
        "high": 0.25,
    },

    "n_iter_no_change": {
        "low": 5,
        "high": 100,
    },

    "tol": {
        "low": 1e-10,
        "high": 1e-4,
    },
}

In [ ]:
HGB_FIXED_PARAMS = {
    "loss": "log_loss",
    "warm_start": False,
    "verbose": 0,
    "random_state": RANDOM_STATE,
    "monotonic_cst": None,
}

In [ ]:
def sample_hgb_params(trial):

    space = HGB_SEARCH_SPACE

    params = dict(
        HGB_FIXED_PARAMS
    )

    params["learning_rate"] = (
        trial.suggest_float(
            "learning_rate",
            space["learning_rate"]["low"],
            space["learning_rate"]["high"],
            log=True,
        )
    )

    tree_mode = (
        trial.suggest_categorical(
            "tree_complexity_mode",
            space[
                "tree_complexity_mode"
            ],
        )
    )

    if tree_mode == "leaf_controlled":

        params["max_leaf_nodes"] = (
            trial.suggest_int(
                "max_leaf_nodes",
                space[
                    "max_leaf_nodes"
                ]["low"],
                space[
                    "max_leaf_nodes"
                ]["high"],
                log=True,
            )
        )

        params["max_depth"] = None

    else:

        params["max_leaf_nodes"] = None

        params["max_depth"] = (
            trial.suggest_int(
                "max_depth",
                space[
                    "max_depth"
                ]["low"],
                space[
                    "max_depth"
                ]["high"],
            )
        )

    params["min_samples_leaf"] = (
        trial.suggest_int(
            "min_samples_leaf",
            space[
                "min_samples_leaf"
            ]["low"],
            space[
                "min_samples_leaf"
            ]["high"],
            log=True,
        )
    )

    params["l2_regularization"] = (
        trial.suggest_float(
            "l2_regularization",
            space[
                "l2_regularization"
            ]["low_nonzero"],
            space[
                "l2_regularization"
            ]["high_nonzero"],
            log=True,
        )
    )

    params["max_features"] = (
        trial.suggest_float(
            "max_features",
            space[
                "max_features"
            ]["low"],
            space[
                "max_features"
            ]["high"],
        )
    )

    params["max_bins"] = (
        trial.suggest_int(
            "max_bins",
            space[
                "max_bins"
            ]["low"],
            space[
                "max_bins"
            ]["high"],
        )
    )

    params["interaction_cst"] = (
        trial.suggest_categorical(
            "interaction_cst",
            space[
                "interaction_cst"
            ],
        )
    )

    params["validation_fraction"] = (
        trial.suggest_float(
            "validation_fraction",
            space[
                "validation_fraction"
            ]["low"],
            space[
                "validation_fraction"
            ]["high"],
        )
    )

    params["n_iter_no_change"] = (
        trial.suggest_int(
            "n_iter_no_change",
            space[
                "n_iter_no_change"
            ]["low"],
            space[
                "n_iter_no_change"
            ]["high"],
        )
    )

    params["tol"] = (
        trial.suggest_float(
            "tol",
            space["tol"]["low"],
            space["tol"]["high"],
            log=True,
        )
    )
    return params

In [ ]:
def make_hgb_fold_frames(X_fit, X_valid):
    X_fit = X_fit.copy()
    X_valid = X_valid.copy()

    for col in hpo_num_features:
        X_fit[col] = pd.to_numeric(
            X_fit[col],
            errors="coerce",
        ).astype("float32")

        X_valid[col] = pd.to_numeric(
            X_valid[col],
            errors="coerce",
        ).astype("float32")

    for col in hpo_cat_features:
        categories = pd.unique(
            X_fit[col].dropna()
        )

        dtype = pd.CategoricalDtype(
            categories=categories
        )

        X_fit[col] = X_fit[col].astype(dtype)
        X_valid[col] = X_valid[col].astype(dtype)

    return X_fit[hpo_features], X_valid[hpo_features]

In [ ]:
def hgb_objective(trial):
    params = sample_hgb_params(trial)
    fold_scores = []

    for train_pos, valid_pos in hpo_cv.split(
        X_hpo,
        y_hpo,
    ):
        X_fit = X_hpo.iloc[train_pos]
        X_valid = X_hpo.iloc[valid_pos]

        y_fit = y_hpo[train_pos]
        y_valid = y_hpo[valid_pos]

        X_fit, X_valid = make_hgb_fold_frames(
            X_fit,
            X_valid,
        )

        model = HistGradientBoostingClassifier(
            **params,
            max_iter=500,
            early_stopping=True,
            class_weight="balanced",
            categorical_features="from_dtype",
        )

        model.fit(X_fit, y_fit)

        prob = model.predict_proba(
            X_valid
        )[:, 1]

        fold_scores.append(
            roc_auc_score(y_valid, prob)
        )

    return float(np.mean(fold_scores))

In [ ]:
RUN_HGB_HPO = False
HGB_TRIALS = 200

if RUN_HGB_HPO:
    hgb_study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.RandomSampler(
            seed=RANDOM_STATE
        ),
    )

    hgb_study.optimize(
        hgb_objective,
        n_trials=HGB_TRIALS,
    )

    print("Beste CV ROC-AUC:", hgb_study.best_value)
    hgb_study.best_params
else:
    print(
        "Schutz vor Dauerlauf mit 200 Trials aktiv. "
        "Für einen tatsächlichen Durchlauf RUN_HGB_HPO=True setzen."
    )

### 5.3 HGB-Ergebnisse aus den vollständigen Versuchen

In [ ]:
# Bestparameter aus dem Optuna-Durchlauf rekonstruieren
def get_best_hgb_params(study):

    best = study.best_trial.params

    params = dict(
        HGB_FIXED_PARAMS
    )

    params["learning_rate"] = (
        best["learning_rate"]
    )

    params["min_samples_leaf"] = (
        best["min_samples_leaf"]
    )

    params["l2_regularization"] = (
        best["l2_regularization"]
    )

    params["max_features"] = (
        best["max_features"]
    )

    params["max_bins"] = (
        best["max_bins"]
    )

    params["interaction_cst"] = (
        best["interaction_cst"]
    )

    params["validation_fraction"] = (
        best["validation_fraction"]
    )

    params["n_iter_no_change"] = (
        best["n_iter_no_change"]
    )

    params["tol"] = (
        best["tol"]
    )

    tree_mode = best[
        "tree_complexity_mode"
    ]

    if tree_mode == "leaf_controlled":

        params["max_leaf_nodes"] = (
            best["max_leaf_nodes"]
        )

        params["max_depth"] = None

    else:

        params["max_leaf_nodes"] = None

        params["max_depth"] = (
            best["max_depth"]
        )

    return params

In [ ]:
# Aktuellen Random-Search-Sieger auf Validierungsdaten testen
current_hgb_validation_auc = None

if (
    "hgb_study" in globals()
    and len(hgb_study.trials) > 0
):

    best_hgb_params = (
        get_best_hgb_params(
            hgb_study
        )
    )

    X_hgb_full_train, X_hgb_validation = (
        make_hgb_fold_frames(
            X_hpo,
            validation_df[
                hpo_features
            ].copy(),
        )
    )

    y_hgb_validation = (
        validation_df[TARGET]
        .to_numpy()
    )

    current_hgb_model = (
        HistGradientBoostingClassifier(
            **best_hgb_params,
            max_iter=500,
            early_stopping=True,
            class_weight="balanced",
            categorical_features="from_dtype",
        )
    )

    current_hgb_model.fit(
        X_hgb_full_train,
        y_hpo,
    )

    current_hgb_prob = (
        current_hgb_model
        .predict_proba(
            X_hgb_validation
        )[:, 1]
    )

    current_hgb_validation_auc = (
        roc_auc_score(
            y_hgb_validation,
            current_hgb_prob,
        )
    )

    print(
        "Aktuelle CV ROC-AUC:",
        hgb_study.best_value,
    )

    print(
        "Aktuelle Validation ROC-AUC:",
        current_hgb_validation_auc,
    )

In [ ]:
hgb_results = []

if (
    "hgb_study" in globals()
    and current_hgb_validation_auc
    is not None
):

    hgb_results.append({
        "Experiment":
            "Aktueller Random Search",

        "Kandidat":
            hgb_study.best_trial.number,

        "Durchläufe":
            len(hgb_study.trials),

        "CV ROC-AUC":
            hgb_study.best_value,

        "Validation ROC-AUC":
            current_hgb_validation_auc,

        "Quelle":
            "aktueller Notebook-Lauf",
    })

In [ ]:
hgb_results.extend([
    {
        "Experiment":
            "Random Search (historisch)",
        "Kandidat":
            67,
        "Durchläufe":
            200,
        "CV ROC-AUC":
            0.638988,
        "Validation ROC-AUC":
            0.635251
    },

    {
        "Experiment":
            "NSGA-II (historisch)",
        "Kandidat":
            70,
        "Durchläufe":
            200,
        "CV ROC-AUC":
            0.639457,
        "Validation ROC-AUC":
            0.633
    },

    {
        "Experiment":
            "TPE (historisch)",
        "Kandidat":
            31,
        "Durchläufe":
            200,
        "CV ROC-AUC":
            0.639605,
        "Validation ROC-AUC":
            0.634087
    },
])

In [ ]:
hgb_reference_results = (
    pd.DataFrame(
        hgb_results
    )
)

hgb_reference_results

Der Vergleich zeigt insgesamt nur kleine Unterschiede zwischen den drei HPO-Verfahren. NSGA-II erreichte in der Kreuzvalidierung eine etwas höhere ROC-AUC als Random Search; auch der später validierte TPE-Kandidat lag in einem ähnlichen Bereich. Insgesamt führten die verschiedenen Suchverfahren damit nur zu relativ kleinen Unterschieden in der Modellgüte. Für die spätere Interpretierbarkeitsanalyse wurde der beste Durchlauf verwendet (Random Search Durchlauf 67). Dieses Modell erreichte auf dem festen Validation-Split eine ROC-AUC von 0,635251 und zeigte damit auch dort eine gute Generalisierungsleistung.

## 6. Ensemblemethoden

Nach der Hyperparameteroptimierung wurden für die Gruppe mehrere Ensemble-Methoden probeweise auf meinen stärksten Einzelmodellen getestet, von simplen Mittelwerten bis zu Stacking. Hinsichtlich der Ensemblezusammensetzung wurden die Basismodelle mit Out-of-Fold-Vorhersagen auf dem Traininingsdatensatz selektiert, und erst anschließend als Ensemble auf den Validierungsdaten geprüft, was bsonders beim Stacking für den Meta-Lerner zur Vermeidung von Target-Leaks wichtig ist. Als Ensemblekandidaten wurden 12 Einzelmodelle verwendet, welche vorher auf den Validierungsdaten getestet wurden, auch auf Fehlerüberlappung.

Für die Ensembleanalyse werden wie erwähnt mehrere zuvor ausgewählte Modellkonfigurationen verwendet. Die Modelle werden hier nicht als bereits erzeugte Dateien vorausgesetzt. Stattdessen sollen ihre Konfigurationen in der Variable ensemble_candidates hinterlegt werden. Für jedes Modell werden anschließend innerhalb derselben 5-fold-Struktur OOF-Predictions auf dem Trainingssplit und Predictions auf der Project-Validation erzeugt.

In [ ]:
ENSEMBLE_FOLDS = 5

ensemble_candidates = []

In [ ]:
# Erwartete Struktur:
#
# ensemble_candidates = [
#     {
#         "name": "HGB Random Search #67",
#         "family": "HGB",
#         "params": {...},
#     },
#     {
#         "name": "XGBoost #65",
#         "family": "XGB",
#         "params": {...},
#     },
#     ...
# ]
#
# Für den vollständigen ursprünglichen Versuch
# wurden 12 Einzelmodelle verwendet.

In [ ]:
ensemble_candidates = [
    # Hier die ausgewählten Modellkonfigurationen
    # aus dem HPO-Abschnitt eintragen.
]

In [ ]:
if len(ensemble_candidates) < 2:
    print(
        "Hinweis: Für eine erneute Ausführung der Ensemble-Pipeline müssen mindestens zwei "
        "Modellkonfigurationen hinterlegt werden."
    )

In [ ]:
if len(ensemble_candidates) != 12:
    print(
        "Hinweis: Im vollständigen Ensembleversuch wurden 12 Einzelmodelle verwendet."
    )

In [ ]:
ensemble_cv = StratifiedKFold(
    n_splits=ENSEMBLE_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

y_ensemble = (
    train_df[TARGET]
    .to_numpy()
)

y_validation = (
    validation_df[TARGET]
    .to_numpy()
)

In [ ]:
oof_predictions = {}
validation_predictions = {}

ensemble_fold_id = np.full(
    len(train_df),
    -1,
    dtype=np.int8,
)

In [ ]:
def build_ensemble_model(candidate):

    if candidate["family"] == "HGB":
        return HistGradientBoostingClassifier(
            **candidate["params"]
        )

    if candidate["family"] == "XGB":
        return XGBClassifier(
            **candidate["params"]
        )

    raise ValueError(
        f"Unbekannte Modellfamilie: "
        f"{candidate['family']}"
    )

In [ ]:
for candidate in ensemble_candidates:

    name = candidate["name"]

    oof_pred = np.zeros(
        len(train_df)
    )

    val_fold_predictions = []

    for fold, (
        train_pos,
        valid_pos,
    ) in enumerate(
        ensemble_cv.split(
            train_df,
            y_ensemble,
        )
    ):

        ensemble_fold_id[
            valid_pos
        ] = fold

        X_fit = train_df.iloc[
            train_pos
        ]

        X_valid = train_df.iloc[
            valid_pos
        ]

        X_project_val = (
            validation_df.copy()
        )

        y_fit = y_ensemble[
            train_pos
        ]

        (
            X_fit_model,
            X_valid_model,
            X_project_val_model,
        ) = prepare_ensemble_fold(
            candidate,
            X_fit,
            y_fit,
            X_valid,
            X_project_val,
        )

        model = build_ensemble_model(
            candidate
        )

        model.fit(
            X_fit_model,
            y_fit,
        )

        oof_pred[valid_pos] = (
            model.predict_proba(
                X_valid_model
            )[:, 1]
        )

        val_fold_predictions.append(
            model.predict_proba(
                X_project_val_model
            )[:, 1]
        )

    oof_predictions[name] = (
        oof_pred
    )

    validation_predictions[name] = (
        np.mean(
            val_fold_predictions,
            axis=0,
        )
    )

candidate_names = [
    candidate["name"]
    for candidate
    in ensemble_candidates
]

In [ ]:
oof_pred_matrix = np.column_stack([
    oof_predictions[name]
    for name in candidate_names
])

val_pred_matrix = np.column_stack([
    validation_predictions[name]
    for name in candidate_names
])

In [ ]:
print(
    "Kreuzvalidierungs-Matrix:",
    oof_pred_matrix.shape
)

print(
    "Validatierungs-Matrix:",
    val_pred_matrix.shape
)

### 6.1 Baseline-Ensembles: Durchschnitt, rangbasierter Durchschnitt und gewichteter rangbasierter Durchschnitt

In [ ]:
# Vorbereitungen für die Erzeugung des rangbasierten Durchschnitts
WEIGHT_PAIR_SAMPLE_SIZE = 200_000
WEIGHT_L2 = 1e-4

In [ ]:
rng = np.random.default_rng(
    RANDOM_STATE
)

positive_idx = np.flatnonzero(
    y_train == 1
)

negative_idx = np.flatnonzero(
    y_train == 0
)

In [ ]:
positive_pairs = rng.choice(
    positive_idx,
    size=WEIGHT_PAIR_SAMPLE_SIZE,
    replace=True,
)

negative_pairs = rng.choice(
    negative_idx,
    size=WEIGHT_PAIR_SAMPLE_SIZE,
    replace=True,
)

In [ ]:
def ranking_loss(weights):
    score_difference = (
        rank_differences @ weights
    )

    return (
        np.mean(
            np.logaddexp(
                0.0,
                -score_difference,
            )
        )
        + WEIGHT_L2
        * np.sum(weights ** 2)
    )

In [ ]:
n_models = (
    oof_rank_matrix.shape[1]
)

initial_weights = (
    np.ones(n_models)
    / n_models
)

constraints = {
    "type": "eq",
    "fun": lambda weights:
        weights.sum() - 1.0,
}

bounds = [
    (0.0, 1.0)
    for _ in range(n_models)
]

In [ ]:
weight_result = minimize(
    ranking_loss,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={
        "maxiter": 300,
    },
)

In [ ]:
learned_weights = np.clip(
    weight_result.x,
    0.0,
    None,
)

learned_weights /= (
    learned_weights.sum()
)

In [ ]:
def percentile_ranks(values):
    ranks = rankdata(
        np.asarray(values),
        method="average",
    )

    return (
        (ranks - 1.0)
        / max(1.0, len(ranks) - 1.0)
    )

In [ ]:
oof_rank_matrix = np.column_stack([
    percentile_ranks(
        oof_pred_matrix[:, col]
    )
    for col in range(
        oof_pred_matrix.shape[1]
    )
])

val_rank_matrix = np.column_stack([
    percentile_ranks(
        val_pred_matrix[:, col]
    )
    for col in range(
        val_pred_matrix.shape[1]
    )
])

In [ ]:
rank_differences = (
    oof_rank_matrix[
        positive_pairs
    ]
    - oof_rank_matrix[
        negative_pairs
    ]
)

In [ ]:
def blend_predictions(
    method,
    prediction_matrix,
    rank_matrix,
    model_indices,
    weights=None,
):
    model_indices = np.asarray(
        model_indices,
        dtype=int,
    )

    if method == "probability_mean":
        return (
            prediction_matrix[
                :, model_indices
            ]
            .mean(axis=1)
        )

    if method == "rank_mean":
        return (
            rank_matrix[
                :, model_indices
            ]
            .mean(axis=1)
        )

    if method == "weighted_rank":
        subset_weights = (
            np.asarray(weights)[
                model_indices
            ]
            .copy()
        )

        if subset_weights.sum() <= 1e-15:
            subset_weights = np.ones(
                len(model_indices)
            )

        subset_weights /= (
            subset_weights.sum()
        )

        return (
            rank_matrix[
                :, model_indices
            ]
            @ subset_weights
        )

    raise ValueError(
        f"Unbekannte Methode: {method}"
    )

In [ ]:
model_indices = [0, 1]

probability_prediction = (
    blend_predictions(
        method="probability_mean",
        prediction_matrix=val_pred_matrix,
        rank_matrix=val_rank_matrix,
        model_indices=model_indices,
    )
)

rank_prediction = (
    blend_predictions(
        method="rank_mean",
        prediction_matrix=val_pred_matrix,
        rank_matrix=val_rank_matrix,
        model_indices=model_indices,
    )
)

weighted_rank_prediction = (
    blend_predictions(
        method="weighted_rank",
        prediction_matrix=val_pred_matrix,
        rank_matrix=val_rank_matrix,
        model_indices=model_indices,
        weights=learned_weights,
    )
)

In [ ]:
blend_methods = [
    "probability_mean",
    "rank_mean",
    "weighted_rank",
]

blend_results = []

for method in blend_methods:

    prediction = blend_predictions(
        method=method,
        prediction_matrix=val_pred_matrix,
        rank_matrix=val_rank_matrix,
        model_indices=model_indices,
        weights=learned_weights,
    )

    blend_results.append({
        "Methode": method,
        "ROC-AUC": roc_auc_score(
            y_val,
            prediction,
        ),
    })

blend_results = pd.DataFrame(
    blend_results
)

blend_results

### 6.2 Teilstichproben-Ensembles

Beim Subsample-Ensemble wird dieselbe Modellkonfiguration mehrfach auf leicht unterschiedlichen geschichteten Teilstichproben des Trainingssplits trainiert. Dadurch entsteht Diversität, ohne Modellfamilie oder Hyperparameter zu verändern. Im vollständigen Experiment wurden für das beste HistGradientBoosting- und XGBoost-Basismodell Trainingsanteile von 70 %, 80 % und 90 % untersucht. Für jede Kombination wurden 20 unabhängige Replikate erzeugt. Positive und negative Beobachtungen wurden jeweils anteilig ohne Zurücklegen gezogen, sodass die ursprüngliche Klassenverteilung annähernd erhalten blieb.  Anschließend wurden unterschiedliche Ensemblegrößen untersucht und die Vorhersagen der jeweils ausgewählten Replikate gemittelt. 

In [ ]:
# Einstellungen
RUN_SUBSAMPLE_ENSEMBLE = False

SUBSAMPLE_FRACTIONS = [
    0.70,
    0.80,
    0.90,
]

SUBSAMPLE_REPLICATES = 20

SUBSAMPLE_ENSEMBLE_SIZES = [
    2, 3, 4, 5, 6,
    7, 8, 10, 15, 20,
]

SUBSAMPLE_COMBINATION_DRAWS = 200

SUBSAMPLE_SEEDS = [
    RANDOM_STATE + 1000 + i
    for i in range(
        SUBSAMPLE_REPLICATES
    )
]

In [ ]:
subsample_bases = {
    candidate["family"]: candidate
    for candidate in ensemble_candidates
    if candidate.get(
        "subsample_base",
        False,
    )
}

In [ ]:
# Vorher Best-Modelle definieren
for family in ["HGB", "XGB"]:

    if family not in subsample_bases:
        print(
            f"Kein Subsample-Basismodell "
            f"für {family} hinterlegt."
        )

In [ ]:
# Geschichtete Stichproben ziehen
def stratified_subsample_positions(
    y,
    fraction,
    seed,
):
    y = np.asarray(
        y,
        dtype=np.int8,
    )

    rng = np.random.default_rng(
        seed
    )

    positive = np.flatnonzero(
        y == 1
    )

    negative = np.flatnonzero(
        y == 0
    )

    n_positive = max(
        2,
        int(
            round(
                len(positive)
                * fraction
            )
        ),
    )

    n_negative = max(
        2,
        int(
            round(
                len(negative)
                * fraction
            )
        ),
    )

    selected_positive = (
        rng.choice(
            positive,
            size=n_positive,
            replace=False,
        )
    )

    selected_negative = (
        rng.choice(
            negative,
            size=n_negative,
            replace=False,
        )
    )

    selected = np.concatenate([
        selected_positive,
        selected_negative,
    ])

    rng.shuffle(selected)

    return selected.astype(
        np.int64
    )

In [ ]:
example_positions = (
    stratified_subsample_positions(
        y_train,
        fraction=0.90,
        seed=1042,
    )
)

print(
    "Trainingszeilen:",
    len(example_positions),
)

print(
    "Positive Rate:",
    y_train[
        example_positions
    ].mean(),
)

In [ ]:
# Einzelnes Subsample-Modell trainieren
def build_subsample_model(
    candidate,
    seed,
):
    params = dict(
        candidate["params"]
    )

    params["random_state"] = seed

    family = candidate["family"]

    if family == "HGB":

        return (
            HistGradientBoostingClassifier(
                **params
            )
        )

    if family == "XGB":

        if (
            "n_estimators"
            not in params
        ):
            params["n_estimators"] = (
                candidate[
                    "n_estimators"
                ]
            )

        params.pop(
            "early_stopping_rounds",
            None,
        )

        return XGBClassifier(
            **params
        )

    raise ValueError(
        f"Unbekannte Familie: "
        f"{family}"
    )

In [ ]:
# Replica
def fit_subsample_replica(
    candidate,
    fraction,
    seed,
):
    selected_positions = (
        stratified_subsample_positions(
            y_train,
            fraction,
            seed,
        )
    )

    features = (
        candidate["features"]
    )

    cat_features = (
        candidate["cat_features"]
    )

    num_features = (
        candidate["num_features"]
    )

    (
        X_subsample,
        X_validation,
    ) = make_oof_target_encoded(
        X_fit_raw=(
            train_df.iloc[
                selected_positions
            ][features]
        ),
        y_fit=(
            y_train[
                selected_positions
            ]
        ),
        X_external_raw=(
            validation_df[
                features
            ]
        ),
        feature_order=features,
        cat_columns=cat_features,
        numeric_columns=num_features,
        smoothing=50.0,
        n_splits=5,
        random_state=RANDOM_STATE,
    )

    model = build_subsample_model(
        candidate,
        seed,
    )

    model.fit(
        X_subsample,
        y_train[
            selected_positions
        ],
    )

    probabilities = (
        model.predict_proba(
            X_validation
        )[:, 1]
    )

    return probabilities

In [ ]:
y_train = (
    train_df[TARGET]
    .to_numpy()
)

y_validation = (
    validation_df[TARGET]
    .to_numpy()
)

In [ ]:
subsample_prediction_tables = {}

if RUN_SUBSAMPLE_ENSEMBLE:

    for family, candidate in (
        subsample_bases.items()
    ):

        for fraction in (
            SUBSAMPLE_FRACTIONS
        ):

            replica_predictions = {
                "target": y_validation,
            }

            for replica, seed in enumerate(
                SUBSAMPLE_SEEDS
            ):

                print(
                    family,
                    fraction,
                    f"Replica {replica + 1}/"
                    f"{SUBSAMPLE_REPLICATES}",
                )

                probabilities = (
                    fit_subsample_replica(
                        candidate,
                        fraction,
                        seed,
                    )
                )

                replica_predictions[
                    f"replica_{replica:02d}"
                ] = probabilities

            prediction_table = (
                pd.DataFrame(
                    replica_predictions
                )
            )

            subsample_prediction_tables[
                (
                    family,
                    fraction,
                )
            ] = prediction_table

In [ ]:
def sampled_combinations(
    items,
    m,
    max_draws,
    seed,
):
    items = tuple(items)

    total = math.comb(
        len(items),
        m,
    )

    if total <= max_draws:
        return list(
            itertools.combinations(
                items,
                m,
            )
        )

    rng = np.random.default_rng(
        seed
    )

    combinations = set()

    while (
        len(combinations)
        < max_draws
    ):

        combination = tuple(
            sorted(
                rng.choice(
                    items,
                    size=m,
                    replace=False,
                ).tolist()
            )
        )

        combinations.add(
            combination
        )

    return sorted(
        combinations
    )

In [ ]:
def analyze_subsample_ensembles(
    prediction_table,
    y_true,
    seed,
):
    replica_columns = [
        col
        for col
        in prediction_table.columns
        if col.startswith(
            "replica_"
        )
    ]

    prediction_matrix = (
        prediction_table[
            replica_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    rows = []

    for m in (
        SUBSAMPLE_ENSEMBLE_SIZES
    ):

        combinations = (
            sampled_combinations(
                range(
                    len(
                        replica_columns
                    )
                ),
                m,
                SUBSAMPLE_COMBINATION_DRAWS,
                seed + m,
            )
        )

        auc_scores = []
        best_auc = -np.inf
        best_combination = None

        for combination in (
            combinations
        ):

            probabilities = (
                prediction_matrix[
                    :,
                    list(combination)
                ]
                .mean(axis=1)
            )

            auc = roc_auc_score(
                y_true,
                probabilities,
            )

            auc_scores.append(
                auc
            )

            if auc > best_auc:

                best_auc = auc

                best_combination = (
                    combination
                )

        rows.append({
            "m": m,

            "n_subsets_evaluated":
                len(combinations),

            "auc_best":
                best_auc,

            "auc_mean":
                np.mean(
                    auc_scores
                ),

            "auc_median":
                np.median(
                    auc_scores
                ),

            "auc_std":
                (
                    np.std(
                        auc_scores,
                        ddof=1,
                    )
                    if len(auc_scores) > 1
                    else 0.0
                ),

            "best_replicas": [
                replica_columns[i]
                for i
                in best_combination
            ],
        })

    return pd.DataFrame(
        rows
    )

In [ ]:
# Alle Varianten vergleichen
subsample_results = []

if RUN_SUBSAMPLE_ENSEMBLE:

    for (
        family,
        fraction,
    ), prediction_table in (
        subsample_prediction_tables
        .items()
    ):

        result = (
            analyze_subsample_ensembles(
                prediction_table,
                y_validation,
                seed=(
                    RANDOM_STATE
                    + int(
                        fraction * 1000
                    )
                ),
            )
        )

        result.insert(
            0,
            "fraction",
            fraction,
        )

        result.insert(
            0,
            "family",
            family,
        )

        subsample_results.append(
            result
        )

    subsample_results = (
        pd.concat(
            subsample_results,
            ignore_index=True,
        )
    )

    subsample_results

In [ ]:
if RUN_SUBSAMPLE_ENSEMBLE:

    best_subsample_results = (
        subsample_results.loc[
            subsample_results
            .groupby("family")[
                "auc_best"
            ]
            .idxmax()
        ]
        .reset_index(drop=True)
    )

    best_subsample_results

In [ ]:
# Ursprügnliche Ergebnisse
historical_subsample_results = (
    pd.DataFrame([
        {
            "family": "HGB",
            "fraction": 0.90,
            "m": 3,
            "Validation ROC-AUC":
                0.636722,
        },
        {
            "family": "XGB",
            "fraction": 0.80,
            "m": 3,
            "Validation ROC-AUC":
                0.636159,
        },
    ])
)

### 6.3 Stacking mit logistischer Regression als Metalerner

In [ ]:
# Einstellungen
RUN_STANDARD_STACKING = False

STACKING_K_VALUES = list(
    range(2, 9)
)

STANDARD_STACKING_C = 1.0

META_FOLDS = 5

In [ ]:
print(
    "OOF-Matrix:",
    oof_pred_matrix.shape,
)

print(
    "Validation-Matrix:",
    val_pred_matrix.shape,
)

print(
    "Fold-Verteilung:",
    np.bincount(
        ensemble_fold_id
    ),
)

In [ ]:
def make_logistic_stacker():

    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                C=STANDARD_STACKING_C,
                penalty="l2",
                solver="lbfgs",
                max_iter=150,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

In [ ]:
def evaluate_stacking_subset(
    model_indices,
    crossfit=False,
):
    model_indices = np.asarray(
        model_indices,
        dtype=int,
    )

    fold_scores = []

    if crossfit:
        crossfit_prediction = np.full(
            len(y_ensemble),
            np.nan,
            dtype=np.float32,
        )

    for fold in range(
        META_FOLDS
    ):

        meta_valid = np.flatnonzero(
            ensemble_fold_id == fold
        )

        meta_train = np.flatnonzero(
            ensemble_fold_id != fold
        )

        model = (
            make_logistic_stacker()
        )

        model.fit(
            oof_pred_matrix[
                np.ix_(
                    meta_train,
                    model_indices,
                )
            ],
            y_ensemble[
                meta_train
            ],
        )

        probabilities = (
            model.predict_proba(
                oof_pred_matrix[
                    np.ix_(
                        meta_valid,
                        model_indices,
                    )
                ]
            )[:, 1]
        )

        fold_scores.append(
            roc_auc_score(
                y_ensemble[
                    meta_valid
                ],
                probabilities,
            )
        )

        if crossfit:

            crossfit_prediction[
                meta_valid
            ] = probabilities

    result = {
        "Meta-CV ROC-AUC":
            float(
                np.mean(
                    fold_scores
                )
            ),

        "Meta-CV Std":
            float(
                np.std(
                    fold_scores,
                    ddof=1,
                )
            ),
    }

    if crossfit:

        result[
            "Meta-CV pooled ROC-AUC"
        ] = roc_auc_score(
            y_ensemble,
            crossfit_prediction,
        )

        result[
            "crossfit_prediction"
        ] = crossfit_prediction

    return result

In [ ]:
# Alle Subsets einer Größe testen
def evaluate_stacking_size(k):

    rows = []

    combinations = (
        itertools.combinations(
            range(
                len(
                    candidate_names
                )
            ),
            k,
        )
    )

    for combination in combinations:

        metrics = (
            evaluate_stacking_subset(
                combination
            )
        )

        rows.append({
            "k": k,

            "model_indices":
                combination,

            "members": [
                candidate_names[i]
                for i
                in combination
            ],

            **metrics,
        })

    return (
        pd.DataFrame(rows)
        .sort_values(
            "Meta-CV ROC-AUC",
            ascending=False,
        )
        .reset_index(drop=True)
    )

In [ ]:
standard_stacking_tables = {}

if RUN_STANDARD_STACKING:

    for k in (
        STACKING_K_VALUES
    ):

        print(
            f"Standard Stacking k={k}"
        )

        standard_stacking_tables[
            k
        ] = evaluate_stacking_size(
            k
        )

In [ ]:
standard_stacking_winners = []

if RUN_STANDARD_STACKING:

    for k in (
        STACKING_K_VALUES
    ):

        best = (
            standard_stacking_tables[
                k
            ]
            .iloc[0]
        )

        model_indices = tuple(
            best[
                "model_indices"
            ]
        )

        meta_cv = (
            evaluate_stacking_subset(
                model_indices,
                crossfit=True,
            )
        )

        stacker = (
            make_logistic_stacker()
        )

        stacker.fit(
            oof_pred_matrix[
                :,
                model_indices
            ],
            y_ensemble,
        )

        validation_probability = (
            stacker.predict_proba(
                val_pred_matrix[
                    :,
                    model_indices
                ]
            )[:, 1]
        )

        validation_auc = (
            roc_auc_score(
                y_validation,
                validation_probability,
            )
        )

        standard_stacking_winners.append({
            "k": k,

            "members":
                best["members"],

            "Meta-CV ROC-AUC":
                meta_cv[
                    "Meta-CV ROC-AUC"
                ],

            "Meta-CV Std":
                meta_cv[
                    "Meta-CV Std"
                ],

            "Meta-CV pooled ROC-AUC":
                meta_cv[
                    "Meta-CV pooled ROC-AUC"
                ],

            "Validation ROC-AUC":
                validation_auc,
        })

    standard_stacking_winners = (
        pd.DataFrame(
            standard_stacking_winners
        )
    )

    standard_stacking_winners

In [ ]:
# Fallback mit ursprünglichen Ergebnissen
historical_standard_stacking = (
    pd.DataFrame({
        "k": [
            2, 3, 4, 5,
            6, 7, 8,
        ],

        "Meta-CV ROC-AUC": [
            0.642260,
            0.642296,
            0.642285,
            0.642304,
            0.642306,
            0.642304,
            0.642283,
        ],

        "Validation ROC-AUC": [
            0.633966,
            0.634250,
            0.634125,
            0.634270,
            0.634442,
            0.634283,
            0.634350,
        ],
    })
)

In [ ]:
if RUN_STANDARD_STACKING:

    standard_stacking_results = (
        standard_stacking_winners
    )

else:

    standard_stacking_results = (
        historical_standard_stacking
        .copy()
    )

In [ ]:
standard_stacking_results

In [ ]:
best_standard_stacking = (
    standard_stacking_results
    .loc[
        standard_stacking_results[
            "Meta-CV ROC-AUC"
        ].idxmax()
    ]
)

best_standard_stacking

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    historical_standard_stacking["k"],
    historical_standard_stacking["Meta-CV ROC-AUC"],
    marker="o",
    label="Meta-CV",
)

ax.plot(
    historical_standard_stacking["k"],
    historical_standard_stacking["Validation ROC-AUC"],
    marker="o",
    label="Project-Validation",
)

ax.set(
    title="Stacking mit LogReg-Metalerner nach Ensemblegröße",
    xlabel="Anzahl Basismodelle k",
    ylabel="ROC-AUC",
)

ax.legend()

plt.tight_layout()
plt.show()

Wie sich zeigt steigt der Meta-Kreuzvalidierungs-Wer nur sehr gering mit der Ensemblegröße an. Nach dem Auswahlkriterium Meta-CV war im ursprünglichen Lauf k=6 der Gewinner. Auf der Project-Validation erreichte diese Variante 0,634. Das liegt leider selbst unter dem einfachen gewichteten Rang und besten Einzelmodellscore.

### 6.4 Greedy Ensemble Selection

In einer späteren Ensemble-Stufe wurde zusätzlich Greedy Ensemble Selection
ausprobiert. Dabei wird schrittweise das Modell ergänzt, das die Out-of-Fold-Leistung der aktuellen Kombination am stärksten verbessert.

In [ ]:
RUN_GREEDY_SELECTION = False

GREEDY_K_VALUES = list(
    range(2, 9)
)

In [ ]:
greedy_candidate_names = (
    candidate_names.copy()
)

greedy_oof_matrix = (
    oof_pred_matrix.copy()
)

greedy_val_matrix = (
    val_pred_matrix.copy()
)

In [ ]:
if RUN_GREEDY_SELECTION:

    best_greedy = (
        greedy_results.loc[
            greedy_results[
                "OOF ROC-AUC"
            ].idxmax()
        ]
    )

    print(
        "Gewähltes k:",
        int(best_greedy["k"]),
    )

    print(
        "OOF ROC-AUC:",
        best_greedy["OOF ROC-AUC"],
    )

    print(
        "Validation ROC-AUC:",
        best_greedy[
            "Validation ROC-AUC"
        ],
    )

    print(
        "Gewichte:",
        best_greedy["weights"],
    )

In [ ]:
def greedy_weights(
    selection_counts
):
    selection_counts = np.asarray(
        selection_counts,
        dtype=float,
    )

    total = (
        selection_counts.sum()
    )

    if total <= 0:
        raise ValueError(
            "Das Ensemble ist leer."
        )

    return (
        selection_counts
        / total
    )

In [ ]:
# Greedy Selection definieren
def greedy_probability_selection(
    oof_matrix,
    val_matrix,
    y_oof,
    candidate_names,
    k_values,
):
    oof_matrix = np.asarray(
        oof_matrix,
        dtype=np.float64,
    )

    val_matrix = np.asarray(
        val_matrix,
        dtype=np.float64,
    )

    n_models = (
        oof_matrix.shape[1]
    )

    counts = np.zeros(
        n_models,
        dtype=int,
    )

    selected_sequence = []

    results = []

    for step in range(
        1,
        max(k_values) + 1,
    ):

        candidate_scores = []

        for candidate_idx in range(
            n_models
        ):

            trial_counts = (
                counts.copy()
            )

            trial_counts[
                candidate_idx
            ] += 1

            trial_weights = (
                greedy_weights(
                    trial_counts
                )
            )

            trial_prediction = (
                oof_matrix
                @ trial_weights
            )

            trial_auc = (
                roc_auc_score(
                    y_oof,
                    trial_prediction,
                )
            )

            candidate_scores.append(
                (
                    candidate_idx,
                    trial_auc,
                )
            )

        best_candidate_idx, _ = (
            max(
                candidate_scores,
                key=lambda item:
                    (
                        item[1],
                        -item[0],
                    ),
            )
        )

        counts[
            best_candidate_idx
        ] += 1

        selected_sequence.append(
            candidate_names[
                best_candidate_idx
            ]
        )

        weights = (
            greedy_weights(
                counts
            )
        )

        oof_prediction = (
            oof_matrix
            @ weights
        )

        val_prediction = (
            val_matrix
            @ weights
        )

        if step in k_values:

            oof_auc = (
                roc_auc_score(
                    y_oof,
                    oof_prediction,
                )
            )

            validation_auc = (
                roc_auc_score(
                    y_validation,
                    val_prediction,
                )
            )

            results.append({
                "k": step,

                "n_unique":
                    int(
                        np.sum(
                            counts > 0
                        )
                    ),

                "OOF ROC-AUC":
                    oof_auc,

                "Validation ROC-AUC":
                    validation_auc,

                "selected_sequence":
                    selected_sequence.copy(),

                "weights":
                    {
                        candidate_names[i]:
                            float(
                                weights[i]
                            )
                        for i
                        in np.flatnonzero(
                            weights > 0
                        )
                    },
            })

    return pd.DataFrame(
        results
    )

In [ ]:
if RUN_GREEDY_SELECTION:

    greedy_results = (
        greedy_probability_selection(
            oof_matrix=(
                greedy_oof_matrix
            ),
            val_matrix=(
                greedy_val_matrix
            ),
            y_oof=y_train,
            candidate_names=(
                greedy_candidate_names
            ),
            k_values=(
                GREEDY_K_VALUES
            ),
        )
    )

    greedy_results

In [ ]:
# Gewinner auswählen
best_greedy = (
    greedy_results.loc[
        greedy_results[
            "OOF ROC-AUC"
        ].idxmax()
    ]
)

best_greedy

In [ ]:
print(
    "Gewähltes k:",
    int(
        best_greedy["k"]
    ),
)

print(
    "OOF ROC-AUC:",
    best_greedy[
        "OOF ROC-AUC"
    ],
)

print(
    "Validation ROC-AUC:",
    best_greedy[
        "Validation ROC-AUC"
    ],
)

print(
    "Gewichte:",
    best_greedy["weights"],
)

In [ ]:
if RUN_GREEDY_SELECTION:

    greedy_results[
        [
            "k",
            "n_unique",
            "OOF ROC-AUC",
            "Validation ROC-AUC",
        ]
    ]

In [ ]:
# Falls RUN_GREEDY_SELECTION=False 
historical_greedy_result = (
    pd.DataFrame([
        {
            "Methode":
                "Greedy ES",
            "Pool":
                "augmented_bags",
            "Blend":
                "prob_replacement",
            "k":
                8,
            "Auswahl":
                "Train-OOF",
            "Validation ROC-AUC":
                0.634509,
        }
    ])
)

In [ ]:
if RUN_GREEDY_SELECTION:
    greedy_development = (
        greedy_results
    )
else:
    greedy_development = (
        historical_greedy_result
        .copy()
    )

greedy_development

## 7. Test-Ergebnisse

Nach Abschluss der Modell-Experimente wurden schließlich mehrere Best-Modelle verschiedener Modell-Familien auf dem Testdatensatz geprüft.

### 7.1 Finale Testergebnisse

In [ ]:
final_test_results = pd.DataFrame([
    [
        "HGB Subsample 90%, m=3",
        0.636722, 0.649203, 0.298407, 0.070400,
        0.233287, 2.332559, 0.637620, 0.660781
    ],
    [
        "Bestes XGBoost",
        0.634709, 0.649191, 0.298383, 0.072799,
        0.235592, 2.355608, 0.637507, 0.661124
    ],
    [
        "Greedy ES k=8",
        0.634553, 0.649009, 0.298017, 0.072049,
        0.233748, 2.337169, 0.637230, 0.661297
    ],
    [
        "Weighted Rank k=2",
        0.635457, 0.648861, 0.297722, 0.071329,
        0.234670, 2.346388, 0.637064, 0.660736
    ],
    [
        "XGB Subsample 80%, m=3",
        0.635992, 0.648500, 0.297000, 0.071068,
        0.232365, 2.323339, 0.636544, 0.660871
    ],
    [
        "Standard Logistic Stacking k=6",
        0.634552, 0.648483, 0.296966, 0.071990,
        0.233748, 2.337169, 0.636622, 0.660741
    ],
    [
        "Regularized Stacking k=5",
        0.634375, 0.648404, 0.296808, 0.071973,
        0.236053, 2.360218, 0.636625, 0.660734
    ],
    [
        "Bestes HGB",
        0.635251, 0.647800, 0.295600, 0.069864,
        0.233748, 2.337169, 0.635994, 0.659747
    ],
    [
        "Logistische Regression",
        0.616662, 0.636681, 0.273363, 0.066984,
        0.219917, 2.198875, 0.625222, 0.648807
    ],
    [
        "MLP (32,16)",
        0.608317, 0.622793, 0.245585, 0.061994,
        0.200092, 2.000653, 0.611401, 0.635278
    ],
], columns=[
    "Modell",
    "Validation ROC-AUC",
    "Test ROC-AUC",
    "Test Normalized Gini",
    "Test Average Precision",
    "Test Recall@Top10%",
    "Test Lift@Top10%",
    "AUC CI 2.5%",
    "AUC CI 97.5%",
])

final_test_results

Der höchste Testwert stammt vom HGB-Subsample-Ensemble mit 0,649203 ROC-AUC.
Das beste einzelne XGBoost liegt mit 0,649191 fast gleichauf. Die Unterschiede zwischen den stärksten Kandidaten sind insgesamt recht klein, weswegen die Modell-Leistungs-Hierarchie vorsichtig interpretiert werden sollte.

### 7.2 Validierung und Test direkt vergleichen

In [ ]:
final_test_results["Test minus Validation"] = (
    final_test_results["Test ROC-AUC"]
    - final_test_results["Validation ROC-AUC"]
)

final_test_results[
    [
        "Modell",
        "Validation ROC-AUC",
        "Test ROC-AUC",
        "Test minus Validation",
    ]
].sort_values(
    "Test ROC-AUC",
    ascending=False,
)

In [ ]:
plot_data = final_test_results.sort_values(
    "Test ROC-AUC"
)

y_pos = np.arange(len(plot_data))
height = 0.36

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.barh(
    y_pos - height / 2,
    plot_data["Validation ROC-AUC"],
    height=height,
    label="Validation",
)

ax.barh(
    y_pos + height / 2,
    plot_data["Test ROC-AUC"],
    height=height,
    label="Test",
)

ax.set_yticks(y_pos)
ax.set_yticklabels(
    plot_data["Modell"]
)

ax.set(
    title="Finaler Vergleich: Validation und interner Test",
    xlabel="ROC-AUC",
)

ax.legend()

plt.tight_layout()
plt.show()

Alle zehn Modelle schneiden auf dem Testsplit besser ab als auf den Validierungsdaten. Dassselbe Phänomen war auf den gruppeninternen Ensembles zu beobachten und könnte ein Hinweis auf eine günstigere Aufteilung im Testdatensplit hindeuten.

### 7.4 Test-AUC mit Bootstrap-Konfidenzintervallen

In [ ]:
plot_data = final_test_results.sort_values(
    "Test ROC-AUC"
)

auc = plot_data["Test ROC-AUC"].to_numpy()
lower = (
    auc
    - plot_data["AUC CI 2.5%"].to_numpy()
)
upper = (
    plot_data["AUC CI 97.5%"].to_numpy()
    - auc
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.barh(
    np.arange(len(plot_data)),
    auc,
    xerr=np.vstack([lower, upper]),
    capsize=3,
)

ax.set_yticks(
    np.arange(len(plot_data))
)

ax.set_yticklabels(
    plot_data["Modell"]
)

ax.set(
    title="Interner Test mit 95%-Bootstrap-Konfidenzintervall",
    xlabel="Test ROC-AUC",
)

plt.tight_layout()
plt.show()

Die Intervalle der stärksten Ensemble- und Boostingmodelle liegen fast
vollständig übereinander. Der Test bestätigt damit eher eine Gruppe ähnlich
starker Modelle als einen einzelnen klar dominierenden Gewinner.

Für eine kompakte Schlussbetrachtung sind vor allem sechs Verfahren
interessant: das HistGradientBoosting Subsample mit der höchsten Test-ROC-AUC, das beste XGBoost mit nahezu gleicher ROC-AUC und höchster Average Precision der Spitzenmodelle. Auch Greedy Ensemble Auswahl war recht nah an den beiden besten Kandidaten dran, Gewichteter Rang war einfach aber ebenfalls robust performant. Die verschiedenen STacking-Varianten waren zwar methodisch interessant ohne jedoch einen klaren Vorteil gegenüber den einfachen Kombinationsmethoden der Einzel-Modell aufzuweisen. Das Ergebnis unterstützt daher auch eher einfache, robuste Ensemblemethoden.

## 8. Gesamtfazit

Die Analyse führt von einer relativ breiten EDA zu einer zunehmend fokussierten
Modellierung. Erkenntnisse aus der EDA:

- Die Daten sind stark unausgeglichen (3,64% gestellter Versicherungsansprüche)
- Es gibt Hinweise auf künstliche oder transformierte Daten
- Missing Values enthalten teilweise Struktur und sollten nicht pauschal entfernt werden. Baumbasierte Modelle können nativ damit umgehen und sind damit interessante Kandidaten für die spätere Modellwahl
- ps_calc_* liefert nur wenig zusätzliches Signal, auch mit den anderen VAriablen
- Ensemble-Boosting-Verfahren sind deutlich stärker als einzelne Entscheidungsbäume oder einfache lineare und distanzbasierte Vergleichs-Baseline-Modelle
- OHE und OOF Target Encoding funktionieren für die stärkeren Modelle gut.

Aus der Hyperparameter-Optimierung:
- HistGradientBoosting und XGBoost sind die wichtigsten Modellfamilien in meinem privaten Modellversuchen gewesen.
- Komplexere Suchverfahren liefern teilweise nur sehr kleine zusätzliche
  CV-Gewinne wie anhand von TPE zu erkennen ist.

Aus den Ensembles:
- Teilstichproben-Ensembles sind überraschend stark.
- Gewichteter Rang ist trotz sehr einfacher Struktur konkurrenzfähig.
- Standard und regularisiertes Stacking verbessern die Validierungsleistung
  nicht gegenüber den einfacheren Varianten.

Aus dem finalen Test:
- HGB Teilstichprobenensemble erreicht mit 0,649203 die höchste ROC-AUC.
- Bestes XGBoost liegt mit 0,649191 fast gleichauf.
- Auch Greedy Ensemble Selection und gewichteter Rang liegen sehr nahe dahinter.
- Die Konfidenzintervalle der stärksten Modelle überlappen deutlich.

Die wichtigste Schlussfolgerung ist deshalb, dass es nicht unbedingt Ensembles, und erst recht keine eher komplexen wie Stacking braucht, um hohe Ergebnisse auf dem Porto-Seguro-Datensatz zu erzielen. Aufgrund der geringen Differenzen am Ende scheint die rein algorithmische Modell- und Ensembleoptimierung auf diesem Datensatz bei einer Test-ROC-AUC von rund 0,649 an eine empirische Leistungsgrenze zu stoßen. Daher bietet es sich für zukünftige Arbeiten an, den Fokus wieder stärker auf die Ebene der Daten zu verlagern, z.B. könnten alternative Architekturen zur Rauschunterdrückung – etwa durch Denoising Autoencoder – oder ein tiefergehendes Reverse-Engineering der anonymisierten Merkmale der Schlüssel sein, um das indirekte Zielsignal in Zukunft noch besser zu entschlüsseln.